# FedEduAI — Validation v19: the differential-privacy arm at 50 rounds

This notebook completes the schedule-length sweep (v17: 5–25 rounds; v18: up to 50 rounds) by measuring the cost of differential privacy directly at 50 rounds instead of inferring it from the control arm E.

What it runs, on the same primary cohort (59,696 students, 12 colleges) with the same seeds and splits:
- The non-private arms (Central, A: FedAvg, E: clipping + Poisson without noise) are trained once at 50 rounds per seed (a deterministic repeat of the v18 run; exact agreement is expected) to provide the paired comparisons.
- Arm D: DP + SecAgg+ (ε = 3, δ = 1e-6) is trained at 50 rounds with three seeds, with the noise multiplier calibrated on the total number of steps of the 50 rounds.
- Paired comparisons: Central − A, A − E, A − D, E − D (paired stratified bootstrap, 300 resamples) and the corrected TOST for Central − A.
- Composition of the privacy budget (RDP) per college over the releases of this run.

Expected time on a Colab CPU: 45–75 minutes (arm D with Opacus is slower than the non-private arms).

Outputs: `schedule_sweep_v19.json`, `dp_budget_composition_v19.json`, `paper_tables_v19.md`, and a zip archive in the results folder.


In [1]:
import sys,subprocess,importlib.util,importlib.metadata
from pathlib import Path
try:
    from google.colab import drive
except ImportError:drive=None
if drive is not None:drive.mount('/content/drive')
requirements={'numpy':'numpy','pandas':'pandas','scipy':'scipy','sklearn':'scikit-learn>=1.3','torch':'torch>=2.6'}
missing=[v for k,v in requirements.items() if importlib.util.find_spec(k) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
for name,version in [('flwr','1.22.0'),('opacus','1.5.4')]:
    try:current=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:current=None
    if current!=version:
        if name in sys.modules:raise RuntimeError('Use a fresh runtime before changing '+name)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',f'{name}=={version}'])


Mounted at /content/drive


In [2]:
# EDIT THIS CELL FOR PATHS / EXPERIMENT POLICY.
from pathlib import Path
from datetime import datetime,timezone,timedelta
import os,re,uuid,json
BASE='/content/drive/MyDrive/iudb'
VERSION='v19'
RUN_TAG=''
RUN_MODE='full'          # full | quick  (quick outputs are NOT publication results)
_stamp=datetime.now(timezone(timedelta(hours=3))).strftime('%Y%m%d_%H%M%S')
_tag=re.sub(r'[^A-Za-z0-9_-]+','_',RUN_TAG).strip('_')
RUN_DIR=str(Path(BASE)/'results'/f'run_{_stamp}_{VERSION}_{uuid.uuid4().hex[:8]}{("_"+_tag) if _tag else ""}')
Path(RUN_DIR).mkdir(parents=True,exist_ok=False)
BOUNDS_CONFIGURED_AT=datetime.now(timezone.utc).isoformat()

CONFIG={
 'paths':{'students':f'{BASE}/students.csv','grades':f'{BASE}/student_records.csv','absence':f'{BASE}/DMO_IU/Absence.csv'},
 'columns':{
  'students':{'student_id':'STUDENT_ID','gpa':'LAST_CUM_GPA','faculty':'FACULTY_NO','degree':'DEGREE_CODE',
              'nationality':'NATIONALITY_CODE','status':'STATUS_CODE','gender':'','study':'','campus':''},
  'grades':{'student_id':'STUDENT_ID','semester':'SEMESTER','mark':'CONFIRMED_MARK','mark_fallback':'MARK','course':'COURSE_NO'},
  'absence':{'student_id':'STUDENT_ID','semester':'SEMESTER','days':'ABSENCE_DAYS','late':'LATE_DAYS'}},
 'settings':{'early_semesters_K':2,'target_source':'final_gpa','target_mode':'gpa_median','pass_mark':60.,
  'gpa_scale_max':5.,'gpa_threshold':None,'risk_status_codes':[],
  'semester_year_offset':1400,       # Explicit IU convention; confirm against source dictionary.
  'min_students_per_faculty':300,'seeds':[0,1,2,3,4],'test_size':.2,'val_size':.2,
  'min_report_n':30,'min_class_n':5,'bootstrap':300,'sample_students':None,'export_predictions':False,
  'equivalence_margins':{'acc':.01,'auc':.005},
  'equivalence_rationale':'Pre-specified before v19 training: one accuracy point and 0.005 AUC are below the smallest difference the institution considers actionable.',
  'output_dir':RUN_DIR}}

WARNING_CFG={'semester_status_path':f'{BASE}/academic_status_by_term.csv',
 'example_threshold':2.75,'policy_scope_confirmed':False,'incident_only':True,'K':2,
 'min_client_n':10 if RUN_MODE=='quick' else 300,'target_recall':.80,
 'source_url':'https://iu.edu.sa/%D8%AF%D9%84%D9%8A%D9%84-%D8%A7%D9%84%D8%B7%D8%A7%D9%84%D8%A8'}

# Training schedule for ALL arms (central baseline uses the same optimizer, batch, passes and initial weights).
PRIVACY_CFG={'seeds':[0] if RUN_MODE=='quick' else [0,1,2,3,4],
 'rounds':1 if RUN_MODE=='quick' else 5,'local_epochs':1,'batch_size':64,
 'learning_rate':.05,'max_grad_norm':1.,'epsilons':[3.] if RUN_MODE=='quick' else [3.,8.],
 'delta':1e-6,'secure_rng':False}

V19_SETTINGS={'target_recall':.8,'bootstrap':30 if RUN_MODE=='quick' else 300,
 # Arms trained from scratch. Central = pooled non-federated baseline with the same schedule.
 'cases':['Central','A_FedAvg','B_SecAgg','C_DP_e3','C_DP_e8','D_DP_SecAgg_e3','D_DP_SecAgg_e8',
          'E_Clipped_Poisson_no_noise','F_Poisson_only_no_clip'],
 'paired_comparisons':[('Central','A_FedAvg'),('A_FedAvg','B_SecAgg'),('A_FedAvg','E_Clipped_Poisson_no_noise'),
          ('F_Poisson_only_no_clip','E_Clipped_Poisson_no_noise'),('A_FedAvg','F_Poisson_only_no_clip'),
          ('E_Clipped_Poisson_no_noise','C_DP_e3'),('E_Clipped_Poisson_no_noise','C_DP_e8'),
          ('C_DP_e3','D_DP_SecAgg_e3'),('A_FedAvg','D_DP_SecAgg_e3')],
 # Offline LiRA: shadow models trained with the SAME training code (task.local_train + FedAvg loop).
 'lira':{'shadows':2 if RUN_MODE=='quick' else 8,'seeds':[0] if RUN_MODE=='quick' else [0,1,2],
         'targets':['A_FedAvg','D_DP_SecAgg_e3'],'max_per_class':2000},
 # Extended cohort: K=1 (students with >=2 observed semesters) as a coverage sensitivity, not the primary claim.
 'extended_cohort':{'enabled':True,'K':1,'cases':['Central','A_FedAvg','D_DP_SecAgg_e3'],
                    'seeds':[0] if RUN_MODE=='quick' else [0,1,2]}}
if RUN_MODE=='quick':
    CONFIG['settings'].update(sample_students=3000,min_students_per_faculty=10,bootstrap=30)
elif RUN_MODE!='full':raise ValueError('RUN_MODE must be full or quick')
# v19: training-schedule sweep. Non-DP arms are trained ONCE at max rounds and evaluated from per-round snapshots;
# DP arms must be retrained per schedule because the noise multiplier is calibrated to the total number of steps.
SWEEP={'rounds':[1] if RUN_MODE=='quick' else [50],
 'seeds':[0] if RUN_MODE=='quick' else [0,1,2],
 'snapshot_cases':['Central','A_FedAvg','E_Clipped_Poisson_no_noise'],   # v19: re-run at 50 rounds (deterministic; expected to reproduce v18 exactly)
 'retrain_cases':['D_DP_SecAgg_e3'],                                     # v19: DP + SecAgg+ (eps = 3) trained at 50 rounds, noise calibrated to 50-round step count
 'question':'At 50 rounds, what is the directly measured DP cost (A minus D) and does it match the clipping-only control (A minus E, E minus D) as it did at 5-25 rounds?'}
V19_SETTINGS['extended_cohort']['enabled']=False
print('Notebook:',f'FedEduAI_Validation_{VERSION}.ipynb')
print('Results:',RUN_DIR)
print('Mode:',RUN_MODE,'| seeds:',PRIVACY_CFG['seeds'],'| arms:',len(V19_SETTINGS['cases']))


Notebook: FedEduAI_Validation_v19.ipynb
Results: /content/drive/MyDrive/iudb/results/run_20260910_190700_v19_c668991d
Mode: full | seeds: [0, 1, 2, 3, 4] | arms: 9


In [3]:
NOTEBOOK_CORE_SHA256='8151b27b0c2901787fd7348e809d5129fd171fd83533fb3a61a4aae25bbaeade'
import os, json, re, gc, copy, hashlib, hmac, secrets, time, math, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd
VERSION = 'v19'
UNKNOWN = 'Unknown / missing'
# Locally keyed pseudonyms: IDs are used to verify splits, never replaced by row numbers.
# This is data minimisation, NOT a formal privacy guarantee. No raw IDs are exported.
# Persistent local secret keeps grouping/order reproducible across notebook restarts.
# The key stays outside the results folder and is never packaged with exports.
_key_path = Path(BASE) / '.fededuai_identity_key'
_key_path.parent.mkdir(parents=True, exist_ok=True)
try:
    fd = os.open(_key_path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
except FileExistsError:
    pass
else:
    with os.fdopen(fd, 'wb') as fh: fh.write(secrets.token_bytes(32))
_ID_SECRET = _key_path.read_bytes()
if len(_ID_SECRET) != 32: raise ValueError('Invalid local identity key length')

def norm_codes(s):
    s = pd.Series(s, index=getattr(s, 'index', None), dtype='string').str.strip()
    s = s.mask(s.str.lower().isin(['', 'nan', 'none', 'null', '<na>', 'nat']))
    return s.str.replace(r'^(\d+)\.0+$', r'\1', regex=True)

def pseudo_ids(s):
    s = norm_codes(s)
    if s.isna().any():
        raise ValueError('Missing identity: filter and count missing IDs before pseudonymisation.')
    mapping={v:hmac.new(_ID_SECRET,v.encode(),hashlib.sha256).hexdigest() for v in s.unique()}
    return s.map(mapping)

def clean_json(x):
    if isinstance(x, dict): return {str(k): clean_json(v) for k,v in x.items()}
    if isinstance(x, (list, tuple, np.ndarray)): return [clean_json(v) for v in x]
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, (float, np.floating)): return float(x) if np.isfinite(x) else None
    if isinstance(x, Path): return str(x)
    return x

def output_path(stem, suffix='.json'):
    if not re.fullmatch(r'[A-Za-z0-9_-]+', stem): raise ValueError('Unsafe output stem')
    if not re.search(r'_v19(?:_|$)', Path(RUN_DIR).name): raise ValueError('Expected v19 run directory')
    return Path(RUN_DIR) / f'{stem}_{VERSION}{suffix}'

def save_result(stem, data):
    p = output_path(stem); payload = {'schema_version': VERSION, 'run_id': Path(RUN_DIR).name, **data}
    tmp = p.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(clean_json(payload), indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(tmp, p); print('Saved:', p.name); return p

def save_figure(stem, fig):
    p=output_path(stem,'.png'); fig.savefig(p,dpi=300,bbox_inches='tight'); return p

def summary(v):
    a=np.asarray([x for x in v if x is not None and np.isfinite(x)],dtype=float)
    return {'mean':float(a.mean()) if len(a) else None, 'sd':float(a.std(ddof=1)) if len(a)>1 else None,
            'n_valid':len(a), 'sd_ddof':1}

def digest_ids(ids):
    return hashlib.sha256('\n'.join(sorted(map(str,ids))).encode()).hexdigest()

def read_table(path, usecols=None, chunk_filter=None, chunksize=250_000):
    ext=Path(path).suffix.lower()
    if ext in ('.xlsx','.xls'): d=pd.read_excel(path,usecols=usecols)
    elif ext in ('.parquet','.pq'): d=pd.read_parquet(path,columns=usecols)
    else:
        for enc in ('utf-8-sig','cp1256','latin-1'):
            try:
                it=pd.read_csv(path,sep=None,engine='python',encoding=enc,usecols=usecols,
                               chunksize=chunksize,on_bad_lines='error',dtype=str)
                parts=[chunk_filter(c) if chunk_filter else c for c in it]
                return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
            except UnicodeDecodeError: continue
        raise ValueError('Could not decode table')
    return chunk_filter(d) if chunk_filter else d

def selected_table(path, columns):
    wanted={c.upper() for c in columns if c}
    d=read_table(path,usecols=lambda c:c.strip().upper() in wanted)
    d.columns=[c.strip().upper() for c in d.columns]
    return d

def semester_year(sem, offset=1400):
    # Explicit source-system convention: final digit is term; short year + configured offset.
    n=pd.to_numeric(sem,errors='coerce'); year=np.floor(n/10)
    return year.where(year>=100,year+offset)

class TrainPreprocessor:
    """Medians, missing indicators, categories and scaling fitted on TRAIN ONLY."""
    def fit(self, frame, numeric, categorical):
        from sklearn.preprocessing import StandardScaler
        self.numeric=list(numeric);self.categorical=list(categorical)
        self.medians=frame[self.numeric].apply(pd.to_numeric,errors='coerce').median().fillna(0)
        self.levels={c:sorted(norm_codes(frame[c]).fillna(UNKNOWN).unique().tolist()) for c in categorical}
        self.names=self.numeric+[c+'__missing' for c in self.numeric]
        for c in categorical: self.names.extend([c+'='+v for v in self.levels[c]]+[c+'=UNSEEN'])
        self.scaler=StandardScaler().fit(self._raw(frame));return self
    def _raw(self, frame):
        n=frame[self.numeric].apply(pd.to_numeric,errors='coerce')
        parts=[n.fillna(self.medians).to_numpy(dtype=np.float32),n.isna().to_numpy(dtype=np.float32)]
        for c in self.categorical:
            s=norm_codes(frame[c]).fillna(UNKNOWN)
            parts.append(np.column_stack([(s==v).to_numpy(dtype=np.float32) for v in self.levels[c]]+
                                         [(~s.isin(self.levels[c])).to_numpy(dtype=np.float32)]))
        return np.column_stack(parts).astype(np.float32)
    def transform(self,frame):
        a=self.scaler.transform(self._raw(frame)).astype(np.float32)
        if not np.isfinite(a).all(): raise ValueError('Non-finite design after TRAIN-only preprocessing')
        return a
    def meta(self):
        return {'feature_names':self.names,'numeric_medians':self.medians.to_dict(),
                'category_levels':self.levels,'fitted_on':'training_partition_only'}

def split_frame(frame, seed, settings, temporal=False):
    from sklearn.model_selection import train_test_split
    if not frame['__ID'].is_unique: raise ValueError('Duplicate real subject IDs in final modeling frame')
    ids=np.arange(len(frame)); test_size=settings.get('test_size',.2);val_size=settings.get('val_size',.2)
    if temporal:
        yr=frame['__YEAR'].to_numpy(float); valid=np.flatnonzero(np.isfinite(yr)); years=np.unique(yr[valid])
        if len(years)<3: raise ValueError('Temporal cohort split requires at least three distinct years')
        cutoff=settings.get('temporal_test_year')
        if cutoff is None:
            selected=[];n=0
            for y in years[::-1]:
                selected.append(y);n+=int((yr==y).sum())
                if n>=len(valid)*test_size:break
            cutoff=min(selected)
        te=valid[yr[valid]>=cutoff];before=valid[yr[valid]<cutoff]
        vy=np.max(yr[before]); va=before[yr[before]==vy];tr=before[yr[before]<vy]
        split_note={'mode':'later_entry_cohort_holdout','test_year_from':float(cutoff),
                    'validation_year':float(vy),'missing_year_excluded':int(len(ids)-len(valid)),
                    'limitation':'Cohort robustness; final GPA/status availability at cutoff is not established.'}
    else:
        # Split before deriving a median label: test outcomes do not set the target threshold.
        pool,te=train_test_split(ids,test_size=test_size,random_state=int(seed))
        tr,va=train_test_split(pool,test_size=val_size,random_state=int(seed)+10000)
        split_note={'mode':'random_subject_holdout','stratified':False}
    if min(len(tr),len(va),len(te))<2:raise ValueError('Insufficient cases in train/validation/test')
    sets=[set(frame.iloc[i]['__ID']) for i in (tr,va,te)]
    overlaps=[len(sets[0]&sets[1]),len(sets[0]&sets[2]),len(sets[1]&sets[2])]
    assert overlaps==[0,0,0]
    split_note.update({'seed':int(seed),'n_train':len(tr),'n_validation':len(va),'n_test':len(te),
                      'identity_overlap_counts':overlaps,'identity_source':'pseudonym_of_source_subject_id',
                      'split_fingerprints':{k:digest_ids(frame.iloc[i]['__ID']) for k,i in zip(('train','validation','test'),(tr,va,te))}})
    return tr,va,te,split_note


In [4]:
def load_institutional(config):
    P,C,S=config['paths'],config['columns'],config['settings'];g=C['grades'];s=C['students'];a=C['absence']
    st=selected_table(P['students'],list(s.values()));flow={'students_input':len(st)}
    sid=s['student_id'].upper(); valid=norm_codes(st[sid]).notna();flow['students_missing_id']=int((~valid).sum());st=st.loc[valid].copy()
    st['__ID']=pseudo_ids(st[sid]); st.drop(columns=sid,inplace=True)
    if not st['__ID'].is_unique:
        raise ValueError('students table has duplicate identities; resolve snapshots explicitly before modeling')
    gr=selected_table(P['grades'],list(g.values()));flow['grade_rows_input']=len(gr)
    sid=g['student_id'].upper();valid=norm_codes(gr[sid]).notna();flow['grade_rows_missing_id']=int((~valid).sum());gr=gr.loc[valid].copy()
    gr['__ID']=pseudo_ids(gr[sid]);gr.drop(columns=sid,inplace=True)
    gr['__SEM']=pd.to_numeric(gr[g['semester'].upper()],errors='coerce')
    mark=pd.to_numeric(gr[g['mark'].upper()],errors='coerce');fb=g.get('mark_fallback','').upper()
    if fb in gr:mark=mark.fillna(pd.to_numeric(gr[fb],errors='coerce'))
    gr['mark']=mark
    valid=mark.between(0,100)&gr['__SEM'].notna()&np.isfinite(gr['__SEM'])&(gr['__SEM']%1==0)
    gr=gr.loc[valid].copy();flow['grade_rows_usable']=len(gr)
    if S.get('sample_students'):
        ids=st['__ID'].sample(min(len(st),int(S['sample_students'])),random_state=0)
        st=st[st['__ID'].isin(ids)];gr=gr[gr['__ID'].isin(ids)]
    gr['mark_sq']=gr['mark']**2;gr['failed']=(gr['mark']<S['pass_mark']).astype(int)
    # Absence reason codes are not days; no unverified not-null=absence indicator is used.
    sem=gr.groupby(['__ID','__SEM'],sort=True).agg(n=('mark','size'),total=('mark','sum'),sq=('mark_sq','sum'),
         low=('mark','min'),high=('mark','max'),failed=('failed','sum')).reset_index()
    sem['mean']=sem['total']/sem['n'];sem['rank']=sem.groupby('__ID')['__SEM'].rank(method='dense').astype(int)
    sem['year']=semester_year(sem['__SEM'],S['semester_year_offset'])
    del gr;gc.collect()
    absence=None
    if P.get('absence'):
        ab=selected_table(P['absence'],list(a.values()));flow['absence_rows_input']=len(ab)
        sid=a['student_id'].upper();v=norm_codes(ab[sid]).notna();ab=ab[v].copy();ab['__ID']=pseudo_ids(ab[sid]);ab.drop(columns=sid,inplace=True)
        ab['__SEM']=pd.to_numeric(ab[a['semester'].upper()],errors='coerce')
        for src,dst in [('days','abs_sum'),('late','late_sum')]:
            ab[dst]=pd.to_numeric(ab[a[src].upper()],errors='coerce')
            flow[dst+'_negative_values']=int((ab[dst]<0).sum());ab.loc[ab[dst]<0,dst]=np.nan
        ab=ab[ab['__SEM'].notna() & (ab['__SEM']%1==0)]
        grouped=ab.groupby(['__ID','__SEM'])
        absence=grouped[['abs_sum','late_sum']].sum(min_count=1).join(grouped.size().rename('absence_rows')).reset_index()
        # Values sum source records (often course-level). They are NOT deduplicated calendar days.
        del ab;gc.collect()
    return {'students':st,'semesters':sem,'absence':absence,'flow':flow}

def build_institutional(tables,config,K=None,min_history=None,target_source=None):
    S=config['settings'];C=config['columns'];K=int(K or S['early_semesters_K']);source=target_source or S['target_source']
    sem=tables['semesters'];st=tables['students'];early=sem[sem['rank']<=K].copy()
    agg=early.groupby('__ID').agg(n_courses=('n','sum'),total=('total','sum'),sq=('sq','sum'),
        mark_min=('low','min'),mark_max=('high','max'),failed=('failed','sum'),early_last_sem=('__SEM','max'))
    agg['mark_mean']=agg['total']/agg['n_courses'];n=agg['n_courses']
    agg['mark_std']=np.sqrt(((agg['sq']-agg['total']**2/n)/(n-1).where(n>1)).clip(lower=0)).fillna(0)
    agg['fail_rate']=agg['failed']/n
    seq=early.pivot(index='__ID',columns='rank',values='mean').reindex(columns=range(1,K+1))
    seq.columns=[f'sem_mean_{i}' for i in range(1,K+1)];agg=agg.join(seq)
    history=sem.groupby('__ID').agg(n_semesters=('__SEM','nunique'),first_sem=('__SEM','min'))
    agg=agg.join(history);agg=agg[agg['n_semesters']>max(K,int(min_history or 0))]
    numeric=['n_courses','mark_mean','mark_std','mark_min','mark_max','fail_rate']+list(seq.columns)
    if tables['absence'] is not None:
        ea=early[['__ID','__SEM']].merge(tables['absence'],on=['__ID','__SEM'],how='left',validate='one_to_one')
        sums=ea.groupby('__ID')[['abs_sum','late_sum']].sum(min_count=1)
        sums['absence_observed_fraction']=ea['abs_sum'].notna().groupby(ea['__ID']).mean()
        sums['absence_any_record']=ea['absence_rows'].notna().groupby(ea['__ID']).max().astype(float)
        agg=agg.join(sums);numeric+=list(sums.columns)
    # These fields remain metadata and are explicitly excluded from the feature lists.
    fut=sem[sem['rank']>K].groupby('__ID')[['total','n']].sum()
    agg['future_mean']=fut['total']/fut['n']
    d=agg.reset_index().merge(st,on='__ID',validate='one_to_one')
    fcol=C['students']['faculty'].upper();d['__CLIENT']=norm_codes(d[fcol]).fillna(UNKNOWN)
    mode=S['target_mode']
    if source=='future_mean':d['__TARGET']=d['future_mean'];target='future course mean beyond early window'
    elif source=='final_gpa':d['__TARGET']=pd.to_numeric(d[C['students']['gpa'].upper()],errors='coerce');target='final cumulative GPA (overlaps early grades)'
    else:raise ValueError('target_source must be final_gpa or future_mean')
    if mode=='status_risk':
        codes={str(v) for v in S['risk_status_codes']}
        if not codes:raise ValueError('status_risk requires institution-approved status codes')
        status=norm_codes(d[C['students']['status'].upper()]);d['__TARGET']=status.isin(codes).astype(float).where(status.notna());target='snapshot risk status; outcome availability unverified'
    elif source=='final_gpa':d.loc[~d['__TARGET'].between(0.01,S.get('gpa_scale_max',5)), '__TARGET']=np.nan
    before=len(d);d=d[d['__TARGET'].notna()].copy();counts=d['__CLIENT'].value_counts()
    kept=counts[counts>=S['min_students_per_faculty']].index
    excluded=counts[~counts.index.isin(kept)].to_dict();d=d[d['__CLIENT'].isin(kept)].reset_index(drop=True)
    if not d['__ID'].is_unique:raise ValueError('Non-unique subject after merges')
    categorical=[C['students'][k].upper() for k in ('degree','nationality','gender','study','campus') if C['students'].get(k)]
    categorical=[c for c in categorical if c in d]
    for c in categorical:d[c]=norm_codes(d[c]).fillna(UNKNOWN)
    d['__YEAR']=semester_year(d['first_sem'],S['semester_year_offset'])
    meta={**tables['flow'],'n_after_history_and_join':before,'n_valid_target_before_clients':int(counts.sum()),
          'n_modeled':len(d),'unique_students':d['__ID'].nunique(),'duplicate_students':0,
          'clients':d['__CLIENT'].value_counts().to_dict(),'excluded_clients_by_min_size':excluded,
          'K':K,'min_total_semesters_exclusive':max(K,int(min_history or 0)),
          'target_source':source,'target_description':target,'numeric_features':numeric,'categorical_features':categorical,
          'feature_cutoff':'first K observed semesters per subject; absence joined to exact same semesters',
          'absence_unit':'sum of source-record absence/lateness values, not unique calendar days',
          'final_snapshot_covariates':'Degree/nationality are exported snapshot attributes; historical availability must be checked.'}
    if 'abs_sum' in d:meta['absence_missing_students']=int(d['abs_sum'].isna().sum())
    if not len(d):raise ValueError('No modeling rows remain')
    return {'frame':d,'numeric':numeric,'categorical':categorical,'sequence':list(seq.columns),'meta':meta}


def coverage_audit(tables,config,primary,extended=None):
    """Row/student counts at every stage, so the paper can state exactly what was processed vs modeled."""
    S=config['settings'];sem=tables['semesters'];st=tables['students'];flow=tables['flow']
    per_student=sem.groupby('__ID').agg(n_semesters=('__SEM','nunique'),n_rows=('n','sum'))
    joined=st[['__ID']].merge(per_student,on='__ID',how='left')
    hist=joined['n_semesters'].fillna(0).astype(int)
    out={'source_tables':{'students_input':flow.get('students_input'),'students_missing_id':flow.get('students_missing_id'),
            'grade_rows_input':flow.get('grade_rows_input'),'grade_rows_missing_id':flow.get('grade_rows_missing_id'),
            'grade_rows_usable':flow.get('grade_rows_usable'),'absence_rows_input':flow.get('absence_rows_input')},
         'students_with_any_usable_grade_row':int((hist>0).sum()),
         'students_by_observed_semesters':{'0':int((hist==0).sum()),'1':int((hist==1).sum()),'2':int((hist==2).sum()),
                                           '3_or_more':int((hist>=3).sum())},
         'grade_rows_attributed_to_students_table':int(joined['n_rows'].fillna(0).sum()),
         'note':'Feature windows use the first K observed semesters; all later usable rows shape only the final-GPA snapshot held by the institution.'}
    def cohort(data,K):
        d=data['frame'];fr=sem[sem['__ID'].isin(d['__ID'])]
        return {'K':K,'n_modeled':len(d),'share_of_students_input':float(len(d)/max(flow.get('students_input',1),1)),
            'share_of_students_with_grades':float(len(d)/max(int((hist>0).sum()),1)),
            'grade_rows_of_modeled_students_total':int(fr['n'].sum()),
            'grade_rows_inside_feature_window':int(fr[fr['rank']<=K]['n'].sum()),
            'positive_rate':float(d['__TARGET'].mean()),'clients':d['__CLIENT'].value_counts().to_dict(),
            'excluded_clients_by_min_size':data['meta'].get('excluded_clients_by_min_size',{})}
    out['primary_cohort']=cohort(primary,int(primary['meta']['K']))
    if extended is not None:out['extended_cohort']=cohort(extended,int(extended['meta']['K']))
    return out


In [5]:
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix, f1_score, brier_score_loss, log_loss,
    average_precision_score, precision_score, recall_score, precision_recall_curve)

def classification_metrics(y,p,threshold=.5):
    y=np.asarray(y,int);p=np.asarray(p,float)
    if not len(y):return {'n':0,'auc':None,'auc_reason':'empty_group'}
    if not np.isfinite(p).all():raise ValueError('Non-finite predictions')
    pred=(p>=threshold).astype(int);tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    ratio=lambda a,b:float(a/b) if b else None
    tpr=ratio(tp,tp+fn);tnr=ratio(tn,tn+fp);auc=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None
    bins=np.minimum((np.clip(p,0,1)*10).astype(int),9);ece=0.;rel=[]
    for b in range(10):
        m=bins==b
        if m.any():
            ece+=m.mean()*abs(p[m].mean()-y[m].mean())
            rel.append({'bin':b,'n':int(m.sum()),'mean_predicted':float(p[m].mean()),'observed_rate':float(y[m].mean())})
    return {'n':len(y),'n_positive':int(y.sum()),'n_negative':int(len(y)-y.sum()),'positive_rate':float(y.mean()),
        'acc':float(accuracy_score(y,pred)),'auc':auc,'auc_reason':None if auc is not None else 'single_class',
        'macro_f1':float(f1_score(y,pred,average='macro',labels=[0,1],zero_division=0)),
        'recall_positive':tpr,'recall_negative':tnr,'fpr':ratio(fp,fp+tn),
        'balanced_accuracy':(tpr+tnr)/2 if tpr is not None and tnr is not None else None,
        'brier':float(brier_score_loss(y,p)),'log_loss':float(log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1])),
        'ece_10_bins':float(ece),'reliability_bins':rel,'threshold':float(threshold),
        'confusion':{'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)},
        'predicted_positive':int(pred.sum()),'constant_prediction':bool(len(np.unique(pred))==1)}

def warning_metrics(y,p,threshold=.5):
    out=classification_metrics(y,p,threshold)
    if not len(y): return out
    out.update(pr_auc_average_precision=float(average_precision_score(y,p)) if np.sum(y)>0 else None,
        precision_positive=float(precision_score(y,np.asarray(p)>=threshold,zero_division=0)),
        f1_positive=float(f1_score(y,np.asarray(p)>=threshold,zero_division=0)))
    return out

def wilson_ci(k,n,z=1.959964):
    if not n:return None
    p=k/n;den=1+z*z/n;centre=(p+z*z/(2*n))/den;half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return [float(max(0,centre-half)),float(min(1,centre+half))]

def threshold_for_recall(y,p,target=.8):
    y=np.asarray(y,int);p=np.asarray(p,float)
    if not (0<target<=1) or not np.isfinite(p).all():raise ValueError('Invalid threshold inputs')
    if len(np.unique(y))<2:raise ValueError('Validation needs both classes for recall-constrained threshold selection')
    _,recall,thresholds=precision_recall_curve(y,p)
    feasible=thresholds[recall[:-1]>=target]
    if not len(feasible):raise ValueError('No feasible validation threshold')
    # Highest feasible threshold minimizes alert burden. Ties are deterministic.
    return float(np.max(feasible))

def groups_for_policy(frame,y,p,threshold,min_n=30,min_class=5):
    out={}
    for g in sorted(frame['__CLIENT'].unique()):
        mask=frame['__CLIENT'].to_numpy()==g
        m=warning_metrics(y[mask],p[mask],threshold)
        m['reportable']=m['n']>=min_n and min(m['n_positive'],m['n_negative'])>=min_class
        c=m['confusion'];m['recall_ci95']=wilson_ci(c['tp'],c['tp']+c['fn']);m['precision_ci95']=wilson_ci(c['tp'],c['tp']+c['fp'])
        m['fpr_ci95']=wilson_ci(c['fp'],c['fp']+c['tn']);m.pop('reliability_bins',None)
        out[str(g)]=m
    overall=warning_metrics(y,p,threshold)
    for k in ['tn','fp','fn','tp']:
        assert sum(m['confusion'][k] for m in out.values())==overall['confusion'][k]
    return out

def fairness_summary(groups):
    rep={g:m for g,m in groups.items() if m['reportable'] and m['recall_positive'] is not None}
    if not rep:return {'status':'no_reportable_clients'}
    rec={g:m['recall_positive'] for g,m in rep.items()};fpr={g:m['fpr'] for g,m in rep.items() if m['fpr'] is not None}
    return {'status':'complete','n_reportable_clients':len(rep),'reportable_clients':sorted(rep),
        'recall_min':min(rec.values()),'recall_max':max(rec.values()),'recall_worst_client':min(rec,key=rec.get),
        'equal_opportunity_gap':float(max(rec.values())-min(rec.values())),
        'fpr_gap':float(max(fpr.values())-min(fpr.values())) if fpr else None,
        'recall_unweighted_mean':float(np.mean(list(rec.values()))),
        'clients_below_half_recall':sorted(g for g,v in rec.items() if v<.5),
        'note':'Gaps are descriptive on held-out test students per faculty; not a causal fairness diagnosis.'}

def paired_case_bootstrap(y,pa,pb,ta,tb,seed,n_boot=300):
    # Conditional on these fitted models/validation thresholds, not training uncertainty.
    rng=np.random.default_rng(seed);classes=[np.flatnonzero(y==c) for c in [0,1]]
    if any(len(i)==0 for i in classes):return {'status':'not_estimable_single_class'}
    def difference(ix):
        a=pa[ix]>=ta;b=pb[ix]>=tb;t=y[ix]
        return [float(np.mean(a==t)-np.mean(b==t)),
            float(a[t==1].mean()-b[t==1].mean()),
            float(a[t==1].sum()/max(a.sum(),1)-b[t==1].sum()/max(b.sum(),1)),
            float(roc_auc_score(t,pa[ix])-roc_auc_score(t,pb[ix])),
            float(average_precision_score(t,pa[ix])-average_precision_score(t,pb[ix])),
            float(np.mean((pa[ix]-t)**2)-np.mean((pb[ix]-t)**2))]
    stats=np.array([difference(np.concatenate([rng.choice(i,len(i),replace=True) for i in classes])) for _ in range(n_boot)])
    keys=['accuracy','recall','precision','roc_auc','average_precision','brier']
    point=difference(np.arange(len(y)))
    return {'status':'complete','n_boot':n_boot,'method':'class-stratified paired test-student bootstrap; conditional on fitted models and validation thresholds',
      'direction':'first_case_minus_second_case',
      'metrics':{k:{'difference':point[i],'ci95':np.quantile(stats[:,i],[.025,.975]).tolist()} for i,k in enumerate(keys)},
      'limitation':'Seeds reuse students; these are per-seed intervals, not independent-study evidence or causal effects.'}

def corrected_tost(diffs,test_train_ratio,margin):
    """Nadeau–Bengio corrected resampled t / TOST over seeds (overlapping random splits)."""
    from scipy.stats import t
    d=np.asarray([v for v in diffs if v is not None],float);n=len(d)
    rec={'n_valid':n,'margin':margin,'mean_diff':float(d.mean()) if n else None}
    if n<2 or np.std(d,ddof=1)==0:
        rec.update(status='insufficient_runs_or_zero_variance',equivalent=None);return rec
    se=float(np.sqrt((1/n+test_train_ratio)*np.var(d,ddof=1)));m=d.mean()
    plo=float(t.sf((m+margin)/se,n-1));phi=float(t.cdf((m-margin)/se,n-1))
    rec.update(status='corrected_resampled_tost',se=se,test_train_ratio=test_train_ratio,
        ci90=(m+np.array([-1,1])*t.ppf(.95,n-1)*se).tolist(),ci95=(m+np.array([-1,1])*t.ppf(.975,n-1)*se).tolist(),
        p_lower=plo,p_upper=phi,equivalent=bool(max(plo,phi)<.05))
    return rec


In [6]:
# Prospective warning where longitudinal status exists; explicit final-GPA proxy otherwise (unchanged from v14/v15).
from sklearn.metrics import roc_curve

def build_warning_data(tables,config,wcfg,K=None):
    path=Path(wcfg['semester_status_path'])
    if not path.is_file():
        # Explicitly labelled sensitivity, not an administrative warning label.
        c=copy.deepcopy(config);c['settings']['min_students_per_faculty']=wcfg['min_client_n']
        data=build_institutional(tables,c,K=K,target_source='final_gpa')
        data['frame']['__TARGET']=(data['frame']['__TARGET']<wcfg['example_threshold']).astype(int)
        data['categorical']=[]
        data['meta'].update(target_source='final_gpa_threshold_proxy',
            target_description='Illustrative final-snapshot low-GPA proxy; NOT next-term official warning',
            threshold=wcfg['example_threshold'],policy_scope_confirmed=False,
            official_warning_experiment='blocked_missing_longitudinal_status',
            missing_file=str(path),categorical_features=[],
            limitation='2.75 example is documented for IU computing; its applicability to all included records is unverified.')
        return data
    a=pd.read_csv(path,dtype=str)
    a.columns=a.columns.str.strip().str.upper()
    required=['STUDENT_ID','SEMESTER','CUM_GPA','FACULTY_NO','IS_REGULAR']
    if not set(required)<=set(a): raise ValueError('academic_status_by_term.csv requires '+', '.join(required))
    a['__ID']=pseudo_ids(a['STUDENT_ID']);a['__SEM']=pd.to_numeric(a['SEMESTER'],errors='raise')
    a['CUM_GPA']=pd.to_numeric(a['CUM_GPA'],errors='raise')
    if not a['CUM_GPA'].between(0,5).all():raise ValueError('Semester GPA must be on validated 0–5 scale')
    regular=pd.to_numeric(a['IS_REGULAR'],errors='raise')
    if not regular.isin([0,1]).all():raise ValueError('IS_REGULAR must be 0/1; no inferred summer mapping')
    if a.duplicated(['__ID','__SEM']).any():raise ValueError('Duplicate student/semester status rows')
    a=a.loc[regular==1].sort_values(['__ID','__SEM']).copy()
    a['__RANK']=a.groupby('__ID').cumcount()+1
    if 'WARNING_FLAG' in a:
        a['__WARN']=pd.to_numeric(a['WARNING_FLAG'],errors='raise')
        if not a['__WARN'].isin([0,1]).all():raise ValueError('WARNING_FLAG must be official 0/1; map source codes explicitly')
        label='recorded_next_regular_term_warning';confirmed=True
    elif 'WARNING_THRESHOLD' in a and wcfg['policy_scope_confirmed']:
        t=pd.to_numeric(a['WARNING_THRESHOLD'],errors='raise')
        if not t.between(.01,5).all():raise ValueError('Invalid policy thresholds')
        a['__WARN']=(a['CUM_GPA']<t).astype(int)
        label='next_regular_term_policy_gpa_risk';confirmed=True
    else:
        raise ValueError('Supply official WARNING_FLAG, or per-row WARNING_THRESHOLD and confirm policy applicability; no automatic 2.75 assignment')
    K=int(wcfg['K']);early=a[a['__RANK']<=K];cut=a[a['__RANK']==K].copy();outcome=a[a['__RANK']==K+1]
    seq=early.pivot(index='__ID',columns='__RANK',values='CUM_GPA').reindex(columns=range(1,K+1))
    seq.columns=[f'gpa_term_{k}' for k in range(1,K+1)]
    d=cut[['__ID','__SEM','FACULTY_NO','CUM_GPA','__WARN']].rename(columns={
        '__SEM':'feature_cutoff_sem','CUM_GPA':'current_gpa','__WARN':'current_warning'})
    d=d.merge(outcome[['__ID','__SEM','__WARN']],on='__ID',validate='one_to_one').rename(
        columns={'__WARN':'__TARGET','__SEM':'outcome_sem'})
    n_at_cutoff=len(cut);before=len(d)
    if wcfg['incident_only']:
        ever=early.groupby('__ID')['__WARN'].max();d=d[d['__ID'].map(ever)==0].copy()
    d=d.merge(seq,on='__ID',validate='one_to_one')
    # Grades are joined to the exact early regular terms, not the first K marked terms.
    e=early[['__ID','__SEM']].merge(tables['semesters'],on=['__ID','__SEM'],how='left',validate='one_to_one')
    z=e.groupby('__ID').agg(total=('total',lambda x:x.sum(min_count=1)),
         n_courses=('n',lambda x:x.sum(min_count=1)),failed=('failed',lambda x:x.sum(min_count=1)))
    z['mark_mean']=z['total']/z['n_courses'];z['fail_rate']=z['failed']/z['n_courses']
    d=d.merge(z[['n_courses','mark_mean','fail_rate']],on='__ID',how='left',validate='one_to_one')
    d['__CLIENT']=norm_codes(d['FACULTY_NO']).fillna(UNKNOWN)
    counts=d['__CLIENT'].value_counts();d=d[d['__CLIENT'].isin(counts[counts>=wcfg['min_client_n']].index)].copy()
    d['__YEAR']=semester_year(d['outcome_sem'],config['settings']['semester_year_offset'])
    numeric=['current_gpa']+list(seq.columns)+['n_courses','mark_mean','fail_rate']
    if not d['__ID'].is_unique or not (d['feature_cutoff_sem']<d['outcome_sem']).all():raise ValueError('Invalid warning chronology')
    if not len(d):raise ValueError('Empty eligible warning cohort')
    return {'frame':d.reset_index(drop=True),'numeric':numeric,'categorical':[],
       'sequence':list(seq.columns),'meta':{'target_source':label,'policy_scope_confirmed':confirmed,
       'n_modeled':len(d),'n_at_feature_cutoff':n_at_cutoff,'n_with_followup':before,
       'missing_followup':n_at_cutoff-before,'incident_only':wcfg['incident_only'],
       'numeric_features':numeric,'categorical_features':[],
       'clients':d['__CLIENT'].value_counts().to_dict(),
       'feature_cutoff':'Kth observed regular enrolled term; supplied IS_REGULAR is authoritative',
       'outcome_horizon':'next observed regular enrolled term; enrollment gaps may exist',
       'source_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
       'selection_note':'No follow-up is excluded and counted, never assigned a negative warning.'}}


In [7]:
# v14 public numeric preprocessing and matched Flower protocol comparisons.
def fixed_private_features(frame,numeric):
    parts=[];names=[];ranges={}
    for c in numeric:
        if c.startswith('gpa_term_') or c=='current_gpa': hi=5.
        elif c in ('mark_mean','mark_std','mark_min','mark_max') or c.startswith('sem_mean_'):hi=100.
        elif c=='n_courses':hi=100.
        elif c in ('fail_rate','absence_observed_fraction','absence_any_record'):hi=1.
        elif c in ('abs_sum','late_sum'):hi=200.
        else:raise ValueError('Declare a public range for feature '+c)
        v=pd.to_numeric(frame[c],errors='coerce').to_numpy(float);miss=~np.isfinite(v)
        z=np.where(miss,0.,np.clip(v,0,hi)/hi)
        parts.extend([z,miss.astype(float)]);names.extend([c,c+'__missing']);ranges[c]=[0,hi]
    return np.column_stack(parts).astype(np.float32),{'method':'public fixed numeric clipping/scaling; zero imputation plus missing flags',
       'ranges':ranges,'names':names,'learned_population_statistics':False,'categorical_features_used':False}


def membership_audit(y_member,p_member,y_nonmember,p_nonmember,seed):
    from sklearn.metrics import roc_auc_score,roc_curve
    rng=np.random.default_rng(seed);ys=[];scores=[]
    # Match class composition so the attack cannot merely exploit different class prevalence.
    for c in (0,1):
        a=np.flatnonzero(y_member==c);b=np.flatnonzero(y_nonmember==c);n=min(len(a),len(b),2000)
        if n<20:continue
        a=rng.choice(a,n,replace=False);b=rng.choice(b,n,replace=False)
        pa=np.clip(p_member[a],1e-7,1-1e-7);pb=np.clip(p_nonmember[b],1e-7,1-1e-7)
        sa=np.log(pa) if c else np.log(1-pa);sb=np.log(pb) if c else np.log(1-pb)
        ys.extend([np.ones(n),np.zeros(n)]);scores.extend([sa,sb])
    if not ys:return {'status':'not_estimable_insufficient_support'}
    labels=np.concatenate(ys);score=np.concatenate(scores);fpr,tpr,_=roc_curve(labels,score)
    auc=float(roc_auc_score(labels,score));boot=[]
    for _ in range(200):
        ix=np.r_[rng.choice(np.flatnonzero(labels==0),int((labels==0).sum()),replace=True),
                 rng.choice(np.flatnonzero(labels==1),int((labels==1).sum()),replace=True)]
        boot.append(roc_auc_score(labels[ix],score[ix]))
    return {'status':'complete','attack':'fixed negative cross-entropy; class-matched membership audit',
      'n_members':int(labels.sum()),'n_nonmembers':int((labels==0).sum()),'auc':auc,
      'auc_ci95':np.quantile(boot,[.025,.975]).tolist(),
      'tpr_at_fpr_le_0_01':float(np.max(tpr[fpr<=.01])),
      'interpretation':'Limited attack baseline, not LiRA and not proof of privacy; no attack direction tuned on test.'}




def attack_metrics(labels,score,seed=0,n_boot=200):
    from sklearn.metrics import roc_auc_score,roc_curve
    labels=np.asarray(labels,int);score=np.asarray(score,float);rng=np.random.default_rng(seed)
    fpr,tpr,_=roc_curve(labels,score);auc=float(roc_auc_score(labels,score));boot=[]
    for _ in range(n_boot):
        ix=np.r_[rng.choice(np.flatnonzero(labels==0),int((labels==0).sum()),replace=True),
                 rng.choice(np.flatnonzero(labels==1),int((labels==1).sum()),replace=True)]
        boot.append(roc_auc_score(labels[ix],score[ix]))
    return {'auc':auc,'auc_ci95':np.quantile(boot,[.025,.975]).tolist(),
        'tpr_at_fpr_le_0_01':float(np.max(tpr[fpr<=.01])) if (fpr<=.01).any() else 0.,
        'tpr_at_fpr_le_0_001':float(np.max(tpr[fpr<=.001])) if (fpr<=.001).any() else 0.,
        'n_members':int(labels.sum()),'n_nonmembers':int((labels==0).sum())}


def logit_confidence(p,y):
    p=np.clip(np.asarray(p,float),1e-7,1-1e-7);py=np.where(y==1,p,1-p)
    return np.log(py/(1-py))


def lira_offline_scores(target_phi,shadow_phi,shadow_in):
    """Offline LiRA (Carlini et al. 2022): per example, Gaussian fit to OUT-shadow confidences.
    target_phi: (n,), shadow_phi: (n_shadows,n), shadow_in: (n_shadows,n) bool."""
    out=~shadow_in;n_out=out.sum(0)
    mu=np.where(n_out>0,(shadow_phi*out).sum(0)/np.maximum(n_out,1),np.nan)
    var=np.where(n_out>1,((shadow_phi-mu)**2*out).sum(0)/np.maximum(n_out-1,1),np.nan)
    # Global variance fallback stabilises examples with few OUT shadows.
    gvar=float(np.nanmean(var)) if np.isfinite(var).any() else 1.
    sd=np.sqrt(np.where(np.isfinite(var)&(var>1e-8),var,gvar))
    mu=np.where(np.isfinite(mu),mu,np.nanmean(shadow_phi))
    from scipy.stats import norm
    return norm.logcdf((target_phi-mu)/sd),n_out


In [8]:
# Flower/Opacus application generated for v19 (identical protocol code to v15 plus: poisson-only control, per-round snapshots).
APP_FILES = {'.gitignore': '__pycache__/\n*.pyc\n*.npz\n*.key\n*.pub\ncertificates/\nprivate*/\nmodels/\n*.fab\n*.log\n', 'fededuai_privacy/__init__.py': '"""FedEduAI privacy/deployment experiments v19."""\n', 'fededuai_privacy/client_app.py': '"""Official Flower SecAgg+ mod protects arrays; ordinary metrics are not masked."""\nimport json\nimport numpy as np\nfrom flwr.client import ClientApp, NumPyClient\nfrom flwr.client.mod import secaggplus_mod\nfrom flwr.common import ConfigRecord\nfrom .task import load_shard,local_train,run_cfg,make_net,set_weights,probabilities\n\nclass StudentClient(NumPyClient):\n    def __init__(self,context):\n        self.context=context\n        self.pid=int(context.node_config[\'partition-id\'])\n        self.shard=load_shard(self.pid);self.cfg=run_cfg(context)\n    def fit(self,parameters,config):\n        key=\'student-dp-accountant\'\n        prev=self.context.state.config_records.get(key)\n        history=json.loads(prev[\'history\']) if prev else []\n        seen=int(prev[\'round\']) if prev else 0\n        rnd=int(config[\'server_round\'])\n        if rnd!=seen+1: raise RuntimeError(\'Unexpected round/retry: stop rather than reset DP accounting\')\n        self.cfg.update(partition=self.pid,round_number=rnd)\n        arrays,history,stats=local_train(parameters,self.shard[\'x_train\'],self.shard[\'y_train\'],self.cfg,history)\n        arrays=[np.clip(a,-float(self.context.run_config[\'clipping-range\']),float(self.context.run_config[\'clipping-range\'])) for a in arrays]\n        self.context.state.config_records[key]=ConfigRecord({\'history\':json.dumps(history),\'round\':rnd})\n        # No losses, labels, identifiers or individual predictions in fit metrics.\n        metrics={\'epsilon\':stats[\'epsilon\'] or 0.,\'seconds\':stats[\'seconds\'],\'steps\':stats[\'steps\']}\n        return arrays,len(self.shard[\'y_train\']),metrics\n    def evaluate(self,parameters,config):\n        m=make_net(self.shard[\'x_train\'].shape[1],self.cfg[\'seed\']);set_weights(m,parameters)\n        y=self.shard[\'y_test\'];p=probabilities(m,self.shard[\'x_test\']);pred=p>=.5\n        if not len(y):return 0.,0,{}\n        return float(np.mean(-(y*np.log(np.clip(p,1e-7,1))+(1-y)*np.log(np.clip(1-p,1e-7,1))))),len(y),{\n            \'tp\':int(((y==1)&pred).sum()),\'fp\':int(((y==0)&pred).sum()),\n            \'tn\':int(((y==0)&~pred).sum()),\'fn\':int(((y==1)&~pred).sum())}\n\ndef client_fn(context):return StudentClient(context).to_client()\ndef configured_secagg_mod(message,context,call_next):\n    if context.run_config[\'secagg\']:\n        return secaggplus_mod(message,context,call_next)\n    return call_next(message,context)\n\napp=ClientApp(client_fn=client_fn,mods=[configured_secagg_mod])\n', 'fededuai_privacy/local_grid.py': '"""Single-process transport of REAL Flower SecAgg+ messages, NOT distributed deployment.\nSerialization counts are protocol-payload bytes, excluding TCP/TLS/retransmissions.\nLocal host can access all data: security evidence here is protocol execution only.\n"""\nimport time,uuid\nfrom flwr.server import Grid\nfrom flwr.common import Context,Message,RecordDict\nfrom flwr.common.typing import Run\nfrom flwr.app.metadata import Metadata\nfrom flwr.common.serde import message_to_proto,message_from_proto\nfrom .client_app import app as client_app\n\nclass LocalGrid(Grid):\n    def __init__(self,config):\n        self.config=config;self._run=Run.create_empty(1);self.replies={}\n        self.contexts={i+1:Context(run_id=1,node_id=i+1,node_config={\'partition-id\':i},state=RecordDict(),run_config=config)\n                       for i in range(int(config[\'num-clients\']))}\n        self.payload_bytes={\'server_to_clients\':0,\'clients_to_server\':0}\n        self.exposed_parameter_arrays=0;self.message_count=0\n    @property\n    def run(self):return self._run\n    def set_run(self,run_id):\n        if run_id!=1:raise ValueError(\'LocalGrid supports its single run only\')\n    def get_node_ids(self):return list(self.contexts)\n    def create_message(self,content,message_type,dst_node_id,group_id,ttl=None):\n        return Message(content=content,dst_node_id=dst_node_id,message_type=message_type,group_id=group_id,ttl=ttl)\n    def push_messages(self,messages):\n        ids=[]\n        for m in messages:\n            old=m.metadata;mid=uuid.uuid4().hex\n            meta=Metadata(1,mid,0,old.dst_node_id,\'\',old.group_id,time.time(),old.ttl,old.message_type)\n            message=Message(metadata=meta,content=m.content)\n            wire=message_to_proto(message);self.payload_bytes[\'server_to_clients\']+=wire.ByteSize()\n            reply=client_app(message_from_proto(wire),self.contexts[old.dst_node_id])\n            if reply.has_error():raise RuntimeError(str(reply.error))\n            if reply.has_content():\n                exposed=sum(len(r) for r in reply.content.array_records.values())\n                self.exposed_parameter_arrays+=exposed\n                if self.config[\'secagg\'] and exposed:raise RuntimeError(\'SecAgg client exposed unmasked parameter arrays\')\n            wire=message_to_proto(reply);self.payload_bytes[\'clients_to_server\']+=wire.ByteSize()\n            self.replies[mid]=message_from_proto(wire);self.message_count+=1;ids.append(mid)\n        return ids\n    def pull_messages(self,message_ids):return [self.replies.pop(i) for i in message_ids]\n    def send_and_receive(self,messages,*,timeout=None):return self.pull_messages(self.push_messages(messages))\n\n\ndef simulate(config):\n    from .server_app import app as server_app\n    grid=LocalGrid(config)\n    server_app(grid,Context(run_id=1,node_id=0,node_config={},state=RecordDict(),run_config=config))\n    return {\'accountant_histories\':{str(k):__import__(\'json\').loads(c.state.config_records[\'student-dp-accountant\'][\'history\']) for k,c in grid.contexts.items()},\n        \'transport\':\'single_process_serialized_Flower_messages\',\n        \'payload_bytes\':grid.payload_bytes,\'message_count\':grid.message_count,\n        \'individual_parameter_arrays_visible_at_server_boundary\':grid.exposed_parameter_arrays,\n        \'network_bytes_measured\':False,\'host_isolation\':False,\n        \'warning\':\'Protocol execution audit, not proof against a malicious host or server/client collusion\'}\n', 'fededuai_privacy/server_app.py': "import json,os,time\nfrom pathlib import Path\nimport numpy as np\nfrom flwr.common import ndarrays_to_parameters,parameters_to_ndarrays\nfrom flwr.server import Grid,LegacyContext,ServerApp,ServerConfig\nfrom flwr.server.strategy import FedAvg\nfrom flwr.server.workflow import DefaultWorkflow,SecAggPlusWorkflow\nfrom .task import make_net,weights\n\napp=ServerApp()\n\n@app.main()\ndef main(grid:Grid,context):\n    rc=context.run_config;n=int(rc['num-clients']);rounds=int(rc['num-server-rounds'])\n    if n<3:raise ValueError('At least three clients required for the configured secure aggregation experiment')\n    root=Path(os.environ['FEDEDUAI_SERVER_OUTPUT']);root.mkdir(parents=True,exist_ok=True)\n    tag=str(rc['case-tag'])\n    if not tag.replace('_','').replace('-','').isalnum():raise ValueError('Invalid case tag')\n    target=root/(tag+'.json')\n    if target.exists():raise FileExistsError('Use a fresh case tag/output folder to avoid mixing runs')\n    t0=time.perf_counter();records=[];pending={}\n    def fit_metrics(items):\n        pending.update(max_client_epsilon=max([m.get('epsilon',0.) for _,m in items],default=0.),\n                       client_training_seconds=sum(m.get('seconds',0.) for _,m in items))\n        return dict(pending)\n    def evaluate(server_round,parameters,config):\n        rec={'round':server_round,**pending,'seconds_since_start':time.perf_counter()-t0}\n        records.append(rec)\n        # Global model is saved in a private output directory, not the publication archive.\n        np.savez(root/(tag+'_model.npz'),**{f'p{i}':p for i,p in enumerate(parameters)})\n        # Per-round snapshots allow validation learning curves to be computed outside the server.\n        np.savez(root/(tag+f'_round{server_round}_model.npz'),**{f'p{i}':p for i,p in enumerate(parameters)})\n        result={'status':'complete' if server_round==rounds else 'in_progress',\n                'mode':os.environ.get('FEDEDUAI_EXECUTION_MODE','unspecified'),\n                'secure_aggregation':bool(rc['secagg']),'dp':bool(rc['dp']),'clip_only':bool(rc.get('clip-only',False)),'poisson_only':bool(rc.get('poisson-only',False)),\n                'rounds':records,'config':dict(rc),\n                'warning':'Model-training accounting only; research metrics/baseline models are not DP releases.'}\n        tmp=target.with_suffix('.tmp');tmp.write_text(json.dumps(result,indent=2));tmp.replace(target)\n        return 0.,{}\n    strategy=FedAvg(fraction_fit=1.,fraction_evaluate=0.,min_fit_clients=n,min_available_clients=n,\n        accept_failures=False,initial_parameters=ndarrays_to_parameters(weights(make_net(int(rc['input-dim']),int(rc['seed'])))),\n        on_fit_config_fn=lambda rnd:{'server_round':rnd},fit_metrics_aggregation_fn=fit_metrics,evaluate_fn=evaluate)\n    legacy=LegacyContext(context=context,config=ServerConfig(num_rounds=rounds),strategy=strategy)\n    if bool(rc['secagg']):\n        # Built-in protocol, not a custom masking approximation. Full participation for primary comparison.\n        fit=SecAggPlusWorkflow(num_shares=n,reconstruction_threshold=n-1,\n            max_weight=float(rc['max-weight']),clipping_range=float(rc['clipping-range']))\n        workflow=DefaultWorkflow(fit_workflow=fit)\n    else:workflow=DefaultWorkflow()\n    workflow(grid,legacy)\n    result=json.loads(target.read_text())\n    if result['status']!='complete':raise RuntimeError('Federation did not finish all configured rounds')\n", 'fededuai_privacy/task.py': '"""FedEduAI v19: student-example DP training; Flower 1.22.0 / Opacus 1.5.4.\nSecurity scope: fixed roster/partition, one example per student, trusted local host.\nResearch evaluation metrics and non-DP comparison models are NOT DP releases.\n"""\nimport json, os, time, secrets\nfrom functools import lru_cache\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom opacus import PrivacyEngine\nfrom opacus.accountants import RDPAccountant\nfrom opacus.accountants.utils import get_noise_multiplier\nget_noise_multiplier=lru_cache(maxsize=256)(get_noise_multiplier)\n\n\ndef make_net(dim, seed=0):\n    with torch.random.fork_rng(devices=[]):\n        torch.manual_seed(int(seed))\n        return nn.Sequential(nn.Linear(dim,32), nn.Tanh(), nn.Linear(32,2))\n\n\ndef weights(model):\n    return [p.detach().cpu().numpy().copy() for p in model.state_dict().values()]\n\n\ndef set_weights(model, arrays):\n    state=model.state_dict()\n    if len(state)!=len(arrays): raise ValueError(\'Parameter count mismatch\')\n    for (k, old), a in zip(state.items(), arrays):\n        if tuple(old.shape)!=tuple(a.shape) or not np.isfinite(a).all():\n            raise ValueError(\'Invalid parameter shape/value\')\n        state[k]=torch.as_tensor(a,dtype=old.dtype)\n    model.load_state_dict(state)\n\n\ndef probabilities(model, x):\n    model.eval(); out=[]\n    with torch.no_grad():\n        for start in range(0,len(x),2048):\n            out.append(torch.softmax(model(torch.as_tensor(x[start:start+2048],dtype=torch.float32)),1)[:,1].numpy())\n    return np.concatenate(out) if out else np.empty(0)\n\n\ndef local_train(arrays,x,y,cfg,history=None):\n    """Keep the accountant across rounds; fresh SGD has no hidden optimizer state.\n    DP noise/shuffle seeds are intentionally NOT set from published experiment seeds.\n    secure_mode=False is a research RNG limitation, explicitly recorded.\n    """\n    if len(x)!=len(y) or len(x)<2: raise ValueError(\'Need at least two local students\')\n    if not np.isfinite(x).all() or not np.isin(y,[0,1]).all(): raise ValueError(\'Invalid training examples\')\n    model=make_net(x.shape[1],cfg[\'seed\']);set_weights(model,arrays)\n    optimizer=torch.optim.SGD(model.parameters(),lr=float(cfg[\'learning_rate\']))\n    # Never derive DP sampling/noise randomness from a published experiment seed.\n    sample_generator=torch.Generator().manual_seed(secrets.randbits(63) if cfg[\'dp\'] else\n        int(cfg[\'seed\'])*1000003+int(cfg.get(\'partition\',0))*1009+int(cfg.get(\'round_number\',0)))\n    noise_generator=torch.Generator().manual_seed(secrets.randbits(63)) if cfg[\'dp\'] and not cfg.get(\'secure_rng\',False) else None\n    loader=DataLoader(TensorDataset(torch.as_tensor(x,dtype=torch.float32),torch.as_tensor(y,dtype=torch.long)),\n                      batch_size=min(int(cfg[\'batch_size\']),len(x)),shuffle=True,generator=sample_generator)\n    engine=None;sigma=None;start=time.perf_counter();steps=0\n    use_clipping=bool(cfg[\'dp\'] or cfg.get(\'clip_only\',False) or cfg.get(\'poisson_only\',False))\n    if use_clipping:\n        # Opacus Poisson sampler uses sample_rate=1/len(loader), including the last batch.\n        q=1/len(loader)\n        sigma=get_noise_multiplier(target_epsilon=float(cfg[\'epsilon\']),target_delta=float(cfg[\'delta\']),\n            sample_rate=q,steps=len(loader)*int(cfg[\'rounds\'])*int(cfg[\'local_epochs\']),accountant=\'rdp\')\n        if cfg.get(\'clip_only\',False) or cfg.get(\'poisson_only\',False):sigma=0.0\n        # poisson_only keeps the Opacus Poisson sampler but disables clipping via an unreachable norm bound.\n        clip_norm=1e6 if cfg.get(\'poisson_only\',False) else float(cfg[\'max_grad_norm\'])\n        engine=PrivacyEngine(accountant=\'rdp\',secure_mode=bool(cfg.get(\'secure_rng\',False)))\n        if history:\n            engine.accountant.history=[(float(a),float(b),int(c)) for a,b,c in history]\n        model,optimizer,loader=engine.make_private(module=model,optimizer=optimizer,data_loader=loader,\n            noise_multiplier=sigma,max_grad_norm=clip_norm,poisson_sampling=True,noise_generator=noise_generator)\n    model.train()\n    for _ in range(int(cfg[\'local_epochs\'])):\n        for xb,yb in loader:\n            optimizer.zero_grad(set_to_none=True)\n            # A sum divided by max(n,1) handles valid empty Poisson batches without NaN.\n            loss=nn.functional.cross_entropy(model(xb),yb,reduction=\'sum\')/max(len(yb),1)\n            loss.backward();optimizer.step();steps+=1\n    base=model._module if engine else model\n    hist=engine.accountant.history if engine and cfg[\'dp\'] else []\n    eps=float(engine.get_epsilon(float(cfg[\'delta\']))) if engine and cfg[\'dp\'] else None\n    if eps is not None and eps>float(cfg[\'epsilon\'])+.03:\n        raise RuntimeError(\'Privacy budget exceeded; refuse to export this update\')\n    return weights(base),hist,{\'steps\':steps,\'seconds\':time.perf_counter()-start,\n        \'epsilon\':eps,\'delta\':float(cfg[\'delta\']) if engine else None,\'noise_multiplier\':sigma,\n        \'accountant\':\'rdp\' if engine else None,\'secure_rng\':bool(cfg.get(\'secure_rng\',False))}\n\n\ndef data_root():\n    # Outside the Flower application, so private shards are never included in its FAB.\n    value=os.environ.get(\'FEDEDUAI_DATA_DIR\')\n    if not value: raise RuntimeError(\'Set FEDEDUAI_DATA_DIR to this host\\\'s private data directory\')\n    return Path(value)\n\n\ndef load_shard(partition):\n    with np.load(data_root()/f\'client_{int(partition)}.npz\',allow_pickle=False) as z:\n        return {k:z[k] for k in z.files}\n\n\ndef run_cfg(context):\n    r=context.run_config\n    return {\'seed\':int(r[\'seed\']),\'rounds\':int(r[\'num-server-rounds\']),\n        \'local_epochs\':int(r[\'local-epochs\']),\'batch_size\':int(r[\'batch-size\']),\n        \'learning_rate\':float(r[\'learning-rate\']),\'max_grad_norm\':float(r[\'max-grad-norm\']),\n        \'epsilon\':float(r[\'epsilon\']),\'delta\':float(r[\'delta\']),\n        \'dp\':bool(r[\'dp\']),\'clip_only\':bool(r.get(\'clip-only\',False)),\'poisson_only\':bool(r.get(\'poisson-only\',False)),\n        \'secure_rng\':bool(r[\'secure-rng\'])}\n', 'monitor_command.py': '"""Measure a controlled command; interface bytes are host-wide, not app-attributed."""\nimport argparse,json,subprocess,time\nfrom pathlib import Path\nimport psutil\np=argparse.ArgumentParser();p.add_argument(\'--interface\',required=True);p.add_argument(\'--output\',required=True);p.add_argument(\'command\',nargs=argparse.REMAINDER)\na=p.parse_args();cmd=a.command[1:] if a.command[:1]==[\'--\'] else a.command\nif not cmd:p.error(\'Supply a command after --\')\npath=Path(a.output)\nif path.exists():p.error(\'Output already exists; choose a new name\')\ndef network():\n n=psutil.net_io_counters(pernic=True)\n if a.interface not in n:raise ValueError(\'Unknown network interface\')\n return n[a.interface]\nbefore=network();start=time.perf_counter();peak=0;cpu={};proc=subprocess.Popen(cmd)\ntry:\n while proc.poll() is None:\n  try:\n   family=[psutil.Process(proc.pid)]+psutil.Process(proc.pid).children(recursive=True);rss=0\n   for child in family:\n    try:\n     rss+=child.memory_info().rss;t=child.cpu_times();cpu[child.pid]=t.user+t.system\n    except psutil.Error:pass\n   peak=max(peak,rss)\n  except psutil.Error:pass\n  time.sleep(.25)\nexcept KeyboardInterrupt:\n proc.terminate();proc.wait()\nafter=network()\nresult={\'exit_code\':proc.returncode,\'wall_seconds\':time.perf_counter()-start,\'peak_process_tree_rss_bytes\':peak,\n \'observed_process_cpu_seconds\':sum(cpu.values()),\'interface\':a.interface,\n \'interface_bytes_sent\':after.bytes_sent-before.bytes_sent,\'interface_bytes_recv\':after.bytes_recv-before.bytes_recv,\n \'scope\':\'Host interface counters over command duration; includes other traffic. Use a dedicated interface/host. Short-lived process CPU may be missed.\'}\npath.write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))\nraise SystemExit(proc.returncode or 0)\n', 'provision_pilot.py': '"""Generate fresh pilot certificates/keys OUTSIDE the distributable application."""\nimport argparse,csv,datetime,ipaddress\nfrom pathlib import Path\nfrom cryptography import x509\nfrom cryptography.x509.oid import NameOID\nfrom cryptography.hazmat.primitives import hashes,serialization\nfrom cryptography.hazmat.primitives.asymmetric import ec,rsa\np=argparse.ArgumentParser();p.add_argument(\'--server-host\',required=True);p.add_argument(\'--output\',required=True);p.add_argument(\'--clients\',type=int,default=3)\na=p.parse_args();out=Path(a.output).resolve()\nif out.exists():raise SystemExit(\'Choose a new empty output path; existing keys are never overwritten\')\nout.mkdir(parents=True,mode=0o700)\ndef write(name,data,private=False):\n q=out/name;q.parent.mkdir(exist_ok=True);q.write_bytes(data);q.chmod(0o600 if private else 0o644)\nnow=datetime.datetime.now(datetime.timezone.utc);ca=rsa.generate_private_key(public_exponent=65537,key_size=3072)\nname=x509.Name([x509.NameAttribute(NameOID.COMMON_NAME,\'FedEduAI v19 pilot CA\')])\ncert=x509.CertificateBuilder().subject_name(name).issuer_name(name).public_key(ca.public_key()).serial_number(x509.random_serial_number()).not_valid_before(now-datetime.timedelta(minutes=5)).not_valid_after(now+datetime.timedelta(days=90)).add_extension(x509.BasicConstraints(ca=True,path_length=0),critical=True).sign(ca,hashes.SHA256())\nwrite(\'ca.key\',ca.private_bytes(serialization.Encoding.PEM,serialization.PrivateFormat.PKCS8,serialization.NoEncryption()),True)\nwrite(\'ca.crt\',cert.public_bytes(serialization.Encoding.PEM))\nserver=rsa.generate_private_key(public_exponent=65537,key_size=3072)\ntry:san=x509.IPAddress(ipaddress.ip_address(a.server_host))\nexcept ValueError:san=x509.DNSName(a.server_host)\nsc=x509.CertificateBuilder().subject_name(x509.Name([x509.NameAttribute(NameOID.COMMON_NAME,a.server_host)])).issuer_name(name).public_key(server.public_key()).serial_number(x509.random_serial_number()).not_valid_before(now-datetime.timedelta(minutes=5)).not_valid_after(now+datetime.timedelta(days=60)).add_extension(x509.SubjectAlternativeName([san,x509.DNSName(\'localhost\'),x509.IPAddress(ipaddress.ip_address(\'127.0.0.1\'))]),critical=False).sign(ca,hashes.SHA256())\nwrite(\'server.key\',server.private_bytes(serialization.Encoding.PEM,serialization.PrivateFormat.PKCS8,serialization.NoEncryption()),True);write(\'server.crt\',sc.public_bytes(serialization.Encoding.PEM))\nkeys=[]\nfor i in range(a.clients):\n key=ec.generate_private_key(ec.SECP384R1());pub=key.public_key().public_bytes(serialization.Encoding.OpenSSH,serialization.PublicFormat.OpenSSH)\n write(f\'node{i}.key\',key.private_bytes(serialization.Encoding.PEM,serialization.PrivateFormat.OpenSSH,serialization.NoEncryption()),True)\n write(f\'node{i}.pub\',pub+b\'\\n\');keys.append(pub.decode())\nwith (out/\'allowed_nodes.csv\').open(\'w\',newline=\'\') as f:csv.writer(f).writerow(keys)\nprint(\'Pilot keys generated. Transfer only each node\\\'s own key pair and ca.crt to that node.\')\n', 'pyproject.toml': '[build-system]\nrequires = ["hatchling"]\nbuild-backend = "hatchling.build"\n[project]\nname = "fededuai-privacy-v19"\nversion = "16.0.0"\ndescription = "FedEduAI student-level privacy experiments"\nlicense = "Apache-2.0"\ndependencies = ["flwr==1.22.0", "opacus==1.5.4", "torch>=2.6,<2.15", "numpy>=1.26,<3", "scikit-learn>=1.3"]\n[tool.hatch.build.targets.wheel]\npackages = ["fededuai_privacy"]\n[tool.flwr.app]\npublisher = "fededuai"\n[tool.flwr.app.components]\nserverapp = "fededuai_privacy.server_app:app"\nclientapp = "fededuai_privacy.client_app:app"\n[tool.flwr.app.config]\nnum-server-rounds = 5\nnum-clients = 3\ninput-dim = 14\nlocal-epochs = 1\nbatch-size = 64\nlearning-rate = 0.05\nmax-grad-norm = 1.0\nepsilon = 3.0\ndelta = 0.000001\ndp = false\nclip-only = false\npoisson-only = false\nsecagg = false\nsecure-rng = false\nseed = 0\nmax-weight = 100000.0\nclipping-range = 8.0\ncase-tag = "baseline_seed0"\n[tool.flwr.federations]\ndefault = "local-simulation"\n[tool.flwr.federations.local-simulation]\noptions.num-supernodes = 3\noptions.backend.client-resources.num-cpus = 1\noptions.backend.client-resources.num-gpus = 0.0\n[tool.flwr.federations.saudi-pilot]\naddress = "127.0.0.1:9093"\nroot-certificates = "certificates/ca.crt"\n', 'start_client.sh': '#!/usr/bin/env bash\nset -euo pipefail\n: "${FEDEDUAI_CERT_DIR:?Set this node\'s certificate directory}"\n: "${FEDEDUAI_DATA_DIR:?Set this node\'s private shard directory}"\n: "${FEDEDUAI_SERVER_HOST:?Set server DNS name or IP matching certificate}"\n: "${FEDEDUAI_PARTITION_ID:?Set this node\'s numeric partition ID}"\nexec flower-supernode \\\n --superlink "$FEDEDUAI_SERVER_HOST:9092" \\\n --root-certificates "$FEDEDUAI_CERT_DIR/ca.crt" \\\n --auth-supernode-private-key "$FEDEDUAI_CERT_DIR/node$FEDEDUAI_PARTITION_ID.key" \\\n --auth-supernode-public-key "$FEDEDUAI_CERT_DIR/node$FEDEDUAI_PARTITION_ID.pub" \\\n --node-config "partition-id=$FEDEDUAI_PARTITION_ID" \\\n --clientappio-api-address 127.0.0.1:9094\n', 'start_server.sh': '#!/usr/bin/env bash\nset -euo pipefail\n: "${FEDEDUAI_CERT_DIR:?Set directory containing pilot certificates}"\n: "${FEDEDUAI_SERVER_OUTPUT:?Set a private output directory}"\nexport FEDEDUAI_EXECUTION_MODE=distributed_pilot\nmkdir -p "$FEDEDUAI_SERVER_OUTPUT"\nexec flower-superlink \\\n --ssl-ca-certfile "$FEDEDUAI_CERT_DIR/ca.crt" \\\n --ssl-certfile "$FEDEDUAI_CERT_DIR/server.crt" \\\n --ssl-keyfile "$FEDEDUAI_CERT_DIR/server.key" \\\n --auth-list-public-keys "$FEDEDUAI_CERT_DIR/allowed_nodes.csv" \\\n --fleet-api-address 0.0.0.0:9092 \\\n --control-api-address 127.0.0.1:9093 \\\n --serverappio-api-address 127.0.0.1:9091\n'}
APP_ROOT=Path(BASE)/'fededuai_apps'/Path(RUN_DIR).name/'fededuai_privacy_v19'
APP_ROOT.mkdir(parents=True,exist_ok=False)
for name,text in APP_FILES.items():
    p=APP_ROOT/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(text,encoding='utf-8')
print('v19 app ready:',APP_ROOT)


v19 app ready: /content/drive/MyDrive/iudb/fededuai_apps/run_20260910_190700_v19_c668991d/fededuai_privacy_v19


In [9]:
# v19 core: train every arm from scratch per seed, evaluate under matched policies, run audits.
import sys,tomllib,math,shutil,importlib
from functools import lru_cache

def optional_tf_disabled():
    from contextlib import contextmanager
    @contextmanager
    def scope():
        absent=object();old=sys.modules.get('tensorflow',absent)
        sys.modules['tensorflow']=None
        try:yield
        finally:
            if old is absent:sys.modules.pop('tensorflow',None)
            else:sys.modules['tensorflow']=old
    return scope()

def load_privacy_app(app_root):
    sys.path.insert(0,str(app_root))
    for name in list(sys.modules):
        if name=='fededuai_privacy' or name.startswith('fededuai_privacy.'):sys.modules.pop(name,None)
    task=importlib.import_module('fededuai_privacy.task')
    grid=importlib.import_module('fededuai_privacy.local_grid')
    base_cfg=tomllib.loads((app_root/'pyproject.toml').read_text())['tool']['flwr']['app']['config']
    return task,grid.simulate,base_cfg

CASE_FLAGS={'Central':None,
 'A_FedAvg':{'dp':False,'secagg':False},'B_SecAgg':{'dp':False,'secagg':True},
 'C_DP_e3':{'dp':True,'secagg':False,'epsilon':3.},'C_DP_e8':{'dp':True,'secagg':False,'epsilon':8.},
 'D_DP_SecAgg_e3':{'dp':True,'secagg':True,'epsilon':3.},'D_DP_SecAgg_e8':{'dp':True,'secagg':True,'epsilon':8.},
 'E_Clipped_Poisson_no_noise':{'dp':False,'secagg':False,'clip-only':True},
 'F_Poisson_only_no_clip':{'dp':False,'secagg':False,'poisson-only':True}}

def local_cfg(cfg,seed,partition=0,round_number=1,local_epochs=None):
    return {'seed':int(seed),'rounds':int(cfg['num-server-rounds']),'local_epochs':int(local_epochs or cfg['local-epochs']),
        'batch_size':int(cfg['batch-size']),'learning_rate':float(cfg['learning-rate']),'max_grad_norm':float(cfg['max-grad-norm']),
        'epsilon':float(cfg['epsilon']),'delta':float(cfg['delta']),'dp':bool(cfg['dp']),'clip_only':bool(cfg.get('clip-only',False)),
        'poisson_only':bool(cfg.get('poisson-only',False)),'secure_rng':bool(cfg['secure-rng']),'partition':int(partition),'round_number':int(round_number)}

def train_central(task,cfg,X,y,seed,curve_fn=None,snapshot_dir=None):
    """Pooled non-federated baseline: same initial weights, optimizer, batch size and number of data passes as FedAvg."""
    arrays=task.weights(task.make_net(X.shape[1],seed));passes=int(cfg['num-server-rounds'])*int(cfg['local-epochs']);rounds=[];t0=time.perf_counter()
    for e in range(1,passes+1):
        c=local_cfg(cfg,seed,0,e,local_epochs=1);c.update(dp=False,clip_only=False,poisson_only=False)
        arrays,_,stats=task.local_train(arrays,X,y,c)
        rec={'round':e,'client_training_seconds':stats['seconds'],'seconds_since_start':time.perf_counter()-t0}
        if curve_fn:rec.update(curve_fn(arrays))
        if snapshot_dir is not None:np.savez(Path(snapshot_dir)/(cfg['case-tag']+f'_round{e}_model.npz'),**{f'p{i}':a for i,a in enumerate(arrays)})
        rounds.append(rec)
    return arrays,{'status':'complete','mode':'pooled_central_baseline','secure_aggregation':False,'dp':False,'rounds':rounds,
        'config':{**cfg,'case-tag':cfg['case-tag']},'note':'Central passes = rounds x local epochs; no protocol transport.'}

def python_fedavg(task,cfg,shards,seed,clip_range):
    """Pure-Python FedAvg with the SAME local_train as the Flower client (used for LiRA shadow models only)."""
    dim=next(iter(shards.values()))[0].shape[1];arrays=task.weights(task.make_net(dim,seed));hist={pid:[] for pid in shards}
    n_total=sum(len(y) for _,y in shards.values())
    for r in range(1,int(cfg['num-server-rounds'])+1):
        total=None
        for pid,(x,y) in shards.items():
            upd,h,_=task.local_train(arrays,x,y,local_cfg(cfg,seed,pid,r),hist[pid]);hist[pid]=h
            upd=[np.clip(a,-clip_range,clip_range) for a in upd];w=len(y)/n_total
            total=[a*w for a in upd] if total is None else [t+a*w for t,a in zip(total,upd)]
        arrays=total
    return arrays

def model_probabilities(task,dim,seed,fp,X):
    m=task.make_net(dim,seed)
    with np.load(fp,allow_pickle=False) as z:task.set_weights(m,[z[f'p{i}'] for i in range(len(z.files))])
    return task.probabilities(m,X)

def run_cohort(data,label,cases,seeds,settings,pcfg,app_root,lira=None):
    task,simulate,base_cfg=load_privacy_app(app_root)
    from opacus.accountants import RDPAccountant
    from opacus.accountants.utils import get_noise_multiplier
    noise=lru_cache(maxsize=256)(get_noise_multiplier)
    d=data['frame'];X,pre=fixed_private_features(d,data['numeric']);y=d['__TARGET'].to_numpy(int)
    clipping={}
    for c,(lo,hi) in pre['ranges'].items():
        v=pd.to_numeric(d[c],errors='coerce').to_numpy(float);finite=np.isfinite(v)
        clipping[c]={'valid':int(finite.sum()),'below_range':int((finite&(v<lo)).sum()),'above_range':int((finite&(v>hi)).sum())}
    private=Path(BASE)/'private_v19'/Path(RUN_DIR).name/label;private.mkdir(parents=True,exist_ok=False)
    runs=[];pairs=[];lira_results=[];dp_histories={}
    for seed in seeds:
        tr,va,te,split=split_frame(d,seed,CONFIG['settings'])
        cl_tr=d.iloc[tr]['__CLIENT'].to_numpy();clients=sorted(d['__CLIENT'].unique())
        counts={g:int((cl_tr==g).sum()) for g in clients}
        eligible=[g for g in clients if counts[g]>=2 and len(np.unique(y[tr][cl_tr==g]))==2]
        if len(eligible)<3:raise ValueError('Secure aggregation experiment needs at least three eligible clients')
        ps=private/f'seed{seed}';ps.mkdir();mo=ps/'models';mo.mkdir()
        shards={}
        for pid,g in enumerate(eligible):
            a=tr[cl_tr==g];b=te[d.iloc[te]['__CLIENT'].to_numpy()==g]
            np.savez(ps/f'client_{pid}.npz',x_train=X[a],y_train=y[a],x_test=X[b],y_test=y[b]);shards[pid]=(X[a],y[a])
        os.environ.update(FEDEDUAI_DATA_DIR=str(ps),FEDEDUAI_SERVER_OUTPUT=str(mo),FEDEDUAI_EXECUTION_MODE='single_process_protocol_simulation')
        common={**base_cfg,'num-server-rounds':int(pcfg['rounds']),'num-clients':len(eligible),'input-dim':int(X.shape[1]),
            'local-epochs':int(pcfg['local_epochs']),'batch-size':int(pcfg['batch_size']),'learning-rate':float(pcfg['learning_rate']),
            'max-grad-norm':float(pcfg['max_grad_norm']),'delta':float(pcfg['delta']),'secure-rng':bool(pcfg['secure_rng']),
            'seed':int(seed),'max-weight':float(max(counts[g] for g in eligible)),'clipping-range':8.0,'epsilon':float(pcfg['epsilons'][0]),
            'dp':False,'secagg':False,'clip-only':False,'poisson-only':False}
        def curve(arrays,seed=seed):
            m=task.make_net(X.shape[1],seed);task.set_weights(m,arrays);p=task.probabilities(m,X[va])
            return {'validation_auc':float(roc_auc_score(y[va],p)),'validation_brier':float(brier_score_loss(y[va],p))}
        predictions={};model_files={}
        for case in cases:
            tag=f'{label}_{case}_seed{seed}';cfg={**common,'case-tag':tag}
            if case=='Central':
                print('Training:',tag,flush=True);arrays,record=train_central(task,cfg,X[tr],y[tr],seed,curve)
                fp=mo/(tag+'_model.npz');np.savez(fp,**{f'p{i}':a for i,a in enumerate(arrays)});transport=None
            else:
                cfg.update(CASE_FLAGS[case]);print('Training:',tag,flush=True)
                transport=simulate(cfg);histories=transport.pop('accountant_histories',None)
                if cfg['dp']:
                    for pid,h in histories.items():dp_histories.setdefault(eligible[int(pid)-1],[]).extend([tuple(x) for x in h])
                record=json.loads((mo/(tag+'.json')).read_text());record['transport']=transport
                fp=mo/(tag+'_model.npz')
                for r in record['rounds']:
                    snap=mo/(tag+f"_round{r['round']}_model.npz")
                    if snap.exists():
                        with np.load(snap,allow_pickle=False) as z:r.update(curve([z[f'p{i}'] for i in range(len(z.files))]))
            pv=model_probabilities(task,X.shape[1],seed,fp,X[va]);pt=model_probabilities(task,X.shape[1],seed,fp,X[te]);ptr=model_probabilities(task,X.shape[1],seed,fp,X[tr])
            threshold=threshold_for_recall(y[va],pv,settings['target_recall']);policies={}
            for name,t in [('fixed_0_5',.5),('validation_recall',threshold)]:
                groups=groups_for_policy(d.iloc[te],y[te],pt,t,CONFIG['settings']['min_report_n'],CONFIG['settings']['min_class_n'])
                policies[name]={'threshold':t,'overall':warning_metrics(y[te],pt,t),'clients':groups,'fairness':fairness_summary(groups)}
            fresh={'case':case,'seed':seed,'status':'complete','cohort':label,'model_sha256':hashlib.sha256(fp.read_bytes()).hexdigest(),
               'split':split,'config':cfg,'eligible_clients':[str(g) for g in eligible],'client_train_counts':{str(g):counts[g] for g in eligible},
               'policies':policies,
               'validation_selection':{'target_recall':settings['target_recall'],'achieved':warning_metrics(y[va],pv,threshold),
                   'method':'highest feasible validation threshold with recall >= target; no test tuning'},
               'training_evidence':{'rounds':record['rounds'],'transport':transport,'origin':'new v19 training from scratch'},
               'membership_loss_audit':membership_audit(y[tr],ptr,y[te],pt,seed)}
            if cfg['dp']:
                entries=[]
                for pid,g in enumerate(eligible):
                    n=counts[g];batch=min(int(cfg['batch-size']),n);steps=math.ceil(n/batch);q=1/steps
                    total=steps*int(cfg['local-epochs'])*int(cfg['num-server-rounds'])
                    sigma=noise(target_epsilon=cfg['epsilon'],target_delta=cfg['delta'],sample_rate=q,steps=total,accountant='rdp')
                    acc=RDPAccountant();acc.history=[(sigma,q,total)]
                    entries.append({'client':str(g),'n_train':n,'sample_rate':q,'noise_multiplier':sigma,'optimizer_steps':total,
                        'max_grad_norm':cfg['max-grad-norm'],'epsilon_planned':acc.get_epsilon(cfg['delta'])})
                reported=record['rounds'][-1]['max_client_epsilon']
                fresh['accounting']={'origin':'live Opacus RDP accountant carried across rounds inside each client; planned schedule recomputed for audit',
                    'delta':cfg['delta'],'nominal_epsilon':cfg['epsilon'],'max_client_epsilon_reported':reported,
                    'max_client_epsilon_planned':max(v['epsilon_planned'] for v in entries),'clients':entries}
            else:fresh['accounting']={'epsilon':None,'dp_guarantee':False}
            runs.append(fresh);predictions[case]=(pt,threshold);model_files[case]=fp
            save_result(f'privacy_validation_{label}',{'status':'in_progress','runs':runs})
        for ca,cb in settings['paired_comparisons']:
            if ca not in predictions or cb not in predictions:continue
            pa,ta=predictions[ca];pb,tb=predictions[cb]
            for policy,at,bt in [('fixed_0_5',.5,.5),('validation_recall',ta,tb)]:
                pairs.append({'seed':seed,'cohort':label,'first_case':ca,'second_case':cb,'policy':policy,
                    **paired_case_bootstrap(y[te],pa,pb,at,bt,seed,settings['bootstrap'])})
        if lira and seed in lira['seeds']:
            rng=np.random.default_rng(seed+777)
            for case in lira['targets']:
                if case not in model_files:continue
                cfg={**common,'case-tag':f'{label}_{case}_seed{seed}'};cfg.update(CASE_FLAGS[case] or {})
                # Class-matched evaluation set: members from TRAIN, non-members from TEST of this seed.
                ev=[];lab=[]
                for c in (0,1):
                    a=tr[y[tr]==c];b=te[y[te]==c];n=min(len(a),len(b),int(lira['max_per_class']))
                    ev.extend([rng.choice(a,n,replace=False),rng.choice(b,n,replace=False)]);lab.extend([np.ones(n),np.zeros(n)])
                ev=np.concatenate(ev);lab=np.concatenate(lab).astype(int)
                pool=np.concatenate([tr,te]);shadow_phi=[];shadow_in=[];t0=time.perf_counter()
                for s in range(int(lira['shadows'])):
                    mask=rng.random(len(pool))<.5;members=pool[mask];cl=d.iloc[members]['__CLIENT'].to_numpy()
                    sh={pid:(X[members[cl==g]],y[members[cl==g]]) for pid,g in enumerate(eligible) if (cl==g).sum()>=2 and len(np.unique(y[members[cl==g]]))==2}
                    arrays=python_fedavg(task,cfg,sh,seed*100+s,cfg['clipping-range'])
                    m=task.make_net(X.shape[1],seed);task.set_weights(m,arrays)
                    shadow_phi.append(logit_confidence(task.probabilities(m,X[ev]),y[ev]));inset=np.zeros(len(pool),bool);inset[np.flatnonzero(mask)]=True
                    pos={v:i for i,v in enumerate(pool)};shadow_in.append(np.array([inset[pos[e]] for e in ev]))
                    print(f'  LiRA shadow {s+1}/{lira["shadows"]} for {case} seed {seed}',flush=True)
                target_phi=logit_confidence(model_probabilities(task,X.shape[1],seed,model_files[case],X[ev]),y[ev])
                scores,n_out=lira_offline_scores(target_phi,np.array(shadow_phi),np.array(shadow_in))
                lira_results.append({'seed':seed,'cohort':label,'target_case':case,'status':'complete',
                    'attack':'offline LiRA (Carlini et al. 2022): per-example Gaussian on OUT-shadow logit confidences',
                    'n_shadows':int(lira['shadows']),'shadow_training':'same task.local_train and FedAvg schedule as the target, random 50% of train+test pool, no Flower transport',
                    'mean_out_shadows_per_example':float(np.mean(n_out)),'seconds':time.perf_counter()-t0,
                    'lira':attack_metrics(lab,scores,seed),'loss_baseline':attack_metrics(lab,target_phi,seed),
                    'interpretation':'Empirical membership leakage on this split; not a proof of privacy. DP arms should approach AUC 0.5 and low TPR at 0.1% FPR.'})
                save_result(f'membership_lira_{label}',{'status':'in_progress','results':lira_results})
    keys=['acc','auc','pr_auc_average_precision','recall_positive','precision_positive','brier','ece_10_bins','macro_f1','predicted_positive']
    summaries={}
    for policy in ['fixed_0_5','validation_recall']:
        summaries[policy]={}
        for case in cases:
            rr=[r for r in runs if r['case']==case]
            if not rr:continue
            common_clients=set.intersection(*[{g for g,m in r['policies'][policy]['clients'].items() if m['reportable']} for r in rr])
            summaries[policy][case]={'overall':{k:summary([r['policies'][policy]['overall'][k] for r in rr]) for k in keys},
                'macro_common_clients':{k:summary([float(np.mean([r['policies'][policy]['clients'][g][k] for g in sorted(common_clients)])) if common_clients else None for r in rr]) for k in ['acc','auc','recall_positive','precision_positive']},
                'per_client_recall_mean':{g:summary([r['policies'][policy]['clients'][g]['recall_positive'] for r in rr]) for g in sorted(common_clients)},
                'fairness':{k:summary([r['policies'][policy]['fairness'].get(k) for r in rr]) for k in ['equal_opportunity_gap','recall_min','recall_unweighted_mean','fpr_gap']},
                'common_clients':sorted(common_clients),'n_seeds':len(rr)}
    equivalence={}
    if 'Central' in cases and 'A_FedAvg' in cases:
        ratio=float(np.mean([r['split']['n_test']/r['split']['n_train'] for r in runs if r['case']=='Central']))
        for policy in ['fixed_0_5','validation_recall']:
            equivalence[policy]={}
            for metric,margin in CONFIG['settings']['equivalence_margins'].items():
                diffs=[]
                for seed in seeds:
                    c=[r for r in runs if r['case']=='Central' and r['seed']==seed];a=[r for r in runs if r['case']=='A_FedAvg' and r['seed']==seed]
                    if c and a:diffs.append(c[0]['policies'][policy]['overall'][metric]-a[0]['policies'][policy]['overall'][metric])
                equivalence[policy][metric]=corrected_tost(diffs,ratio,margin)
        equivalence['method']='Nadeau-Bengio corrected resampled paired TOST over seeds; direction central minus federated; margins pre-specified in CONFIG'
    result={'status':'complete','cohort':label,'target':data['meta'],'preprocessing':pre,'feature_clipping_audit':clipping,
       'runs':runs,'summary_by_policy':summaries,'paired_comparisons':pairs,'equivalence_central_vs_fedavg':equivalence,
       'membership_lira':lira_results,'dp_histories_by_client':{str(k):v for k,v in dp_histories.items()},
       'private_artifacts_directory':str(private),'n_training_runs':len(runs),
       'limitations':['Research metrics, Central, A, B, E and F are not differentially private releases.',
          'E changes clipping and Poisson sampling together; F isolates Poisson sampling; E versus C/D isolates added noise under matched settings.',
          'Single-process Flower transport: protocol execution audit, not an independent-host deployment.',
          'Bootstrap is conditional on each trained model; seeds reuse students, so per-seed intervals are not independent studies.',
          'Offline LiRA uses shadows trained without Flower transport; empirical leakage only, not a privacy proof.',
          'The outcome is a final-snapshot low-GPA proxy unless a longitudinal status file is supplied.']}
    save_result(f'privacy_validation_{label}',result);return result

def compose_dp_budget(results,delta):
    """RDP composition of every DP training history per faculty across all seeds, cases and cohorts."""
    from opacus.accountants import RDPAccountant
    merged={}
    for res in results:
        for g,h in res.get('dp_histories_by_client',{}).items():merged.setdefault(g,[]).extend([tuple(x) for x in h])
    out={}
    for g,h in merged.items():
        acc=RDPAccountant();acc.history=[(float(a),float(b),int(c)) for a,b,c in h];out[g]={'n_releases_steps':len(h),'epsilon_composed':float(acc.get_epsilon(delta))}
    return {'delta':delta,'per_client':out,'epsilon_max_composed':max((v['epsilon_composed'] for v in out.values()),default=None),
        'scope':'All DP model releases in this run composed per faculty (RDP). Non-DP arms and evaluation metrics are outside this bound.',
        'note':'Students recur across seeds; per-faculty composition over every release is the conservative bound for a student.'}


In [10]:
packages={}
for name in ['numpy','pandas','scipy','scikit-learn','torch','flwr','opacus']:
    try:packages[name]=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:packages[name]=None
save_result('run_manifest',{'notebook':'FedEduAI_Validation_v19.ipynb','generated_core_sha256':NOTEBOOK_CORE_SHA256,
 'run_mode':RUN_MODE,'settings':V19_SETTINGS,'training':PRIVACY_CFG,'data_config':CONFIG,'warning_config':WARNING_CFG,'packages':packages,
 'purpose':'Train all arms from scratch on the institutional cohort; matched central baseline; controls E/F; fairness, calibration, TOST, LiRA, DP composition, coverage audit.'})


Saved: run_manifest_v19.json


PosixPath('/content/drive/MyDrive/iudb/results/run_20260910_190700_v19_c668991d/run_manifest_v19.json')

In [11]:
print('Reading university tables...',flush=True)
iu_tables=load_institutional(CONFIG)
warning_data=build_warning_data(iu_tables,CONFIG,WARNING_CFG)
extended_data=None
if V19_SETTINGS['extended_cohort']['enabled']:
    extended_data=build_warning_data(iu_tables,CONFIG,WARNING_CFG,K=V19_SETTINGS['extended_cohort']['K'])
    extended_data['meta']['role']='extended cohort sensitivity (K=1); not the primary claim'
save_result('warning_target_audit',warning_data['meta'])
coverage=coverage_audit(iu_tables,CONFIG,warning_data,extended_data)
save_result('coverage_audit',coverage)
print('Primary cohort:',warning_data['meta']['n_modeled'],'students in',len(warning_data['meta']['clients']),'faculties')
if extended_data is not None:print('Extended cohort (K=1):',extended_data['meta']['n_modeled'],'students')
print('Grade rows usable:',coverage['source_tables']['grade_rows_usable'],'| students input:',coverage['source_tables']['students_input'])


Reading university tables...
Saved: warning_target_audit_v19.json
Saved: coverage_audit_v19.json
Primary cohort: 59696 students in 12 faculties
Grade rows usable: 3819767 | students input: 105339


In [12]:
# v19 sweep: schedule length vs. federated gap and DP cost. Reuses v16 functions unchanged.
def run_schedule_sweep(data,sweep,settings,pcfg,app_root):
    task,simulate,base_cfg=load_privacy_app(app_root)
    d=data['frame'];X,pre=fixed_private_features(d,data['numeric']);y=d['__TARGET'].to_numpy(int)
    R_max=max(sweep['rounds']);private=Path(BASE)/'private_v19'/Path(RUN_DIR).name;private.mkdir(parents=True,exist_ok=False)
    records=[];pairs=[];dp_histories={};splits_ratio=[]
    def evaluate(fp,seed,va,te,case,rounds_used,origin):
        pv=model_probabilities(task,X.shape[1],seed,fp,X[va]);pt=model_probabilities(task,X.shape[1],seed,fp,X[te])
        thr=threshold_for_recall(y[va],pv,settings['target_recall'])
        rec={'case':case,'seed':seed,'rounds':rounds_used,'origin':origin,'model_sha256':hashlib.sha256(fp.read_bytes()).hexdigest(),
             'validation_auc':float(roc_auc_score(y[va],pv)),'threshold':thr,
             'fixed_0_5':warning_metrics(y[te],pt,.5),'validation_recall':warning_metrics(y[te],pt,thr)}
        for k in ('fixed_0_5','validation_recall'):rec[k].pop('reliability_bins',None)
        return rec,pt,thr
    for seed in sweep['seeds']:
        tr,va,te,split=split_frame(d,seed,CONFIG['settings']);splits_ratio.append(split['n_test']/split['n_train'])
        cl_tr=d.iloc[tr]['__CLIENT'].to_numpy();clients=sorted(d['__CLIENT'].unique())
        counts={g:int((cl_tr==g).sum()) for g in clients}
        eligible=[g for g in clients if counts[g]>=2 and len(np.unique(y[tr][cl_tr==g]))==2]
        ps=private/f'seed{seed}';ps.mkdir();mo=ps/'models';mo.mkdir()
        for pid,g in enumerate(eligible):
            a=tr[cl_tr==g];b=te[d.iloc[te]['__CLIENT'].to_numpy()==g]
            np.savez(ps/f'client_{pid}.npz',x_train=X[a],y_train=y[a],x_test=X[b],y_test=y[b])
        os.environ.update(FEDEDUAI_DATA_DIR=str(ps),FEDEDUAI_SERVER_OUTPUT=str(mo),FEDEDUAI_EXECUTION_MODE='single_process_protocol_simulation')
        common={**base_cfg,'num-clients':len(eligible),'input-dim':int(X.shape[1]),'local-epochs':int(pcfg['local_epochs']),
            'batch-size':int(pcfg['batch_size']),'learning-rate':float(pcfg['learning_rate']),'max-grad-norm':float(pcfg['max_grad_norm']),
            'delta':float(pcfg['delta']),'secure-rng':bool(pcfg['secure_rng']),'seed':int(seed),'max-weight':float(max(counts[g] for g in eligible)),
            'clipping-range':8.0,'epsilon':float(pcfg['epsilons'][0]),'dp':False,'secagg':False,'clip-only':False,'poisson-only':False}
        def curve(arrays,seed=seed):
            m=task.make_net(X.shape[1],seed);task.set_weights(m,arrays);p=task.probabilities(m,X[va])
            return {'validation_auc':float(roc_auc_score(y[va],p)),'validation_brier':float(brier_score_loss(y[va],p))}
        preds={}   # (case,rounds) -> (pt,thr)
        # --- non-DP arms: one run at R_max, evaluate snapshots ---
        for case in sweep['snapshot_cases']:
            tag=f'sweep_{case}_R{R_max}_seed{seed}';cfg={**common,'num-server-rounds':R_max,'case-tag':tag}
            print('Training:',tag,flush=True)
            if case=='Central':arrays,record=train_central(task,cfg,X[tr],y[tr],seed,curve,snapshot_dir=mo)
            else:
                cfg.update(CASE_FLAGS[case]);transport=simulate(cfg);transport.pop('accountant_histories',None)
                record=json.loads((mo/(tag+'.json')).read_text())
            curves=[]
            for r in range(1,R_max+1):
                snap=mo/(tag+f'_round{r}_model.npz')
                with np.load(snap,allow_pickle=False) as z:curves.append({'round':r,**curve([z[f'p{i}'] for i in range(len(z.files))])})
            best=max(curves,key=lambda c:c['validation_auc'])['round']
            for R in sorted(set(sweep['rounds'])|{best}):
                rec,pt,thr=evaluate(mo/(tag+f'_round{R}_model.npz'),seed,va,te,case,R,'snapshot_of_single_run' if R!=best else 'best_validation_round')
                if R==best:rec['is_best_validation_round']=True
                rec['learning_curve']=curves;records.append(rec);preds[(case,R)]=(pt,thr)
            preds[(case,'best')]=preds[(case,best)];records[-1]['best_round']=best
        # --- DP arm: retrain per schedule length ---
        for case in sweep['retrain_cases']:
            for R in sweep['rounds']:
                tag=f'sweep_{case}_R{R}_seed{seed}';cfg={**common,'num-server-rounds':R,'case-tag':tag};cfg.update(CASE_FLAGS[case])
                print('Training:',tag,flush=True);transport=simulate(cfg);hist=transport.pop('accountant_histories',None)
                for pid,h in hist.items():dp_histories.setdefault(eligible[int(pid)-1],[]).extend([tuple(x) for x in h])
                record=json.loads((mo/(tag+'.json')).read_text())
                curves=[]
                for r in range(1,R+1):
                    snap=mo/(tag+f'_round{r}_model.npz')
                    if snap.exists():
                        with np.load(snap,allow_pickle=False) as z:curves.append({'round':r,**curve([z[f'p{i}'] for i in range(len(z.files))])})
                rec,pt,thr=evaluate(mo/(tag+'_model.npz'),seed,va,te,case,R,'retrained_for_this_schedule')
                rec['learning_curve']=curves;rec['max_client_epsilon_reported']=record['rounds'][-1].get('max_client_epsilon')
                records.append(rec);preds[(case,R)]=(pt,thr)
        # --- paired comparisons per schedule ---
        for R in sweep['rounds']:
            for ca,cb in [('Central','A_FedAvg'),('A_FedAvg','E_Clipped_Poisson_no_noise'),('A_FedAvg','D_DP_SecAgg_e3'),('E_Clipped_Poisson_no_noise','D_DP_SecAgg_e3')]:
                if (ca,R) not in preds or (cb,R) not in preds:continue
                pa,ta=preds[(ca,R)];pb,tb=preds[(cb,R)]
                for policy,at,bt in [('fixed_0_5',.5,.5),('validation_recall',ta,tb)]:
                    pairs.append({'seed':seed,'rounds':R,'first_case':ca,'second_case':cb,'policy':policy,**paired_case_bootstrap(y[te],pa,pb,at,bt,seed,settings['bootstrap'])})
        if ('Central','best') in preds and ('A_FedAvg','best') in preds:
            pa,ta=preds[('Central','best')];pb,tb=preds[('A_FedAvg','best')]
            pairs.append({'seed':seed,'rounds':'best_validation_round','first_case':'Central','second_case':'A_FedAvg','policy':'validation_recall',**paired_case_bootstrap(y[te],pa,pb,ta,tb,seed,settings['bootstrap'])})
        save_result('schedule_sweep',{'status':'in_progress','records':records,'paired_comparisons':pairs})
    # --- summaries + corrected TOST per schedule ---
    ratio=float(np.mean(splits_ratio))
    keys=['acc','auc','recall_positive','precision_positive','brier','predicted_positive']
    summary_tbl={};tost={}
    for R in sweep['rounds']+['best']:
        summary_tbl[str(R)]={}
        for case in sweep['snapshot_cases']+sweep['retrain_cases']:
            rr=[r for r in records if r['case']==case and (r['rounds']==R if R!='best' else r.get('is_best_validation_round'))]
            if not rr:continue
            summary_tbl[str(R)][case]={pol:{k:summary([r[pol][k] for r in rr]) for k in keys} for pol in ('fixed_0_5','validation_recall')}
            summary_tbl[str(R)][case]['validation_auc']=summary([r['validation_auc'] for r in rr])
            summary_tbl[str(R)][case]['best_round']=summary([r.get('best_round') for r in rr if r.get('best_round') is not None]) if R=='best' else None
        tost[str(R)]={}
        for pol in ('fixed_0_5','validation_recall'):
            tost[str(R)][pol]={}
            for metric,margin in CONFIG['settings']['equivalence_margins'].items():
                diffs=[]
                for seed in sweep['seeds']:
                    c=[r for r in records if r['case']=='Central' and r['seed']==seed and (r['rounds']==R if R!='best' else r.get('is_best_validation_round'))]
                    a=[r for r in records if r['case']=='A_FedAvg' and r['seed']==seed and (r['rounds']==R if R!='best' else r.get('is_best_validation_round'))]
                    if c and a:diffs.append(c[0][pol][metric]-a[0][pol][metric])
                tost[str(R)][pol][metric]=corrected_tost(diffs,ratio,margin)
    result={'status':'complete','question':sweep['question'],'target':data['meta'],'sweep':sweep,'records':records,'paired_comparisons':pairs,
        'summary_by_rounds':summary_tbl,'equivalence_central_vs_fedavg_by_rounds':tost,'dp_histories_by_client':{str(k):v for k,v in dp_histories.items()},
        'limitations':['Non-DP snapshots come from a single run per seed; DP arms are retrained per schedule because noise is calibrated to total steps.',
           'Best-validation-round selection uses validation only; test metrics at that round are still conditional on the split.',
           'Seeds reuse students; TOST is corrected approximately (Nadeau-Bengio), not exact.',
           'Central at long schedules may overfit; compare best-validation rounds, not only the final round.']}
    save_result('schedule_sweep',result);return result

with optional_tf_disabled():
    try:
        sweep_result=run_schedule_sweep(warning_data,SWEEP,V19_SETTINGS,PRIVACY_CFG,APP_ROOT)
        composition=compose_dp_budget([sweep_result],PRIVACY_CFG['delta']);save_result('dp_budget_composition',composition)
    except Exception as exc:
        save_result('execution_failure',{'status':'failed','exception_type':type(exc).__name__,'message':str(exc)});raise
print('Sweep complete:',len(sweep_result['records']),'evaluations;',len(sweep_result['paired_comparisons']),'paired comparisons')


Training: sweep_Central_R50_seed0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Training: sweep_A_FedAvg_R50_seed0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      [INIT]
INFO:flwr:[INIT]
INFO :      Using initial global parameters provided by strategy
INFO:flwr:Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO:flwr:Starting evaluation of initial global parameters
INFO :      initial parameters (loss, other metrics): 0.0, {}
INFO:flwr:initial parameters (loss, other metrics): 0.0, {}
INFO :      
INFO:flwr:
INFO :      [ROUND 1]
INFO:flwr:[ROUND 1]
INFO :      configure_fit: strategy sampled 12 clients (out of 12)
INFO:flwr:configure_fit: strategy sampled 12 clients (out of 12)
INFO :      aggregate_fit: received 12 results and 0 failures
INFO:flwr:agg

Training: sweep_E_Clipped_Poisson_no_noise_R50_seed0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      [INIT]
INFO:flwr:[INIT]
INFO :      Using initial global parameters provided by strategy
INFO:flwr:Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO:flwr:Starting evaluation of initial global parameters
INFO :      initial parameters (loss, other metrics): 0.0, {}
INFO:flwr:initial parameters (loss, other metrics): 0.0, {}
INFO :      
INFO:flwr:
INFO :      [ROUND 1]
INFO:flwr:[ROUND 1]
INFO :      configure_fit: strategy sampled 12 clients (out of 12)
INFO:flwr:configure_fit: strategy sampled 12 clients (out of 12)
/usr/local/lib/python3.13/dist-packages/opacus/accountants/analysis/rdp.py:

... (17 outputs left out)

Streaming output truncated to the last 5000 lines.
INFO :      Secure aggregation completed.
INFO:flwr:Secure aggregation completed.
INFO :      fit progress: (11, 0.0, {}, 53.24205392600015)
INFO:flwr:fit progress: (11, 0.0, {}, 53.24205392600015)
INFO :      configure_evaluate: no clients selected, skipping evaluation
INFO:flwr:configure_evaluate: no clients selected, skipping evaluation
INFO :      
INFO:flwr:
INFO :      [ROUND 12]
INFO:flwr:[ROUND 12]
INFO :      Secure aggregation commencing.
INFO:flwr:Secure aggregation commencing.
INFO :      configure_fit: strategy sampled 12 clients (out of 12)
INFO:flwr:configure_fit: strategy sampled 12 clients (out of 12)
DEBUG:flwr:[Stage 0] Sending configurations to 12 clients.
DEBUG:flwr:Node 1: starting stage 0...
DEBUG:flwr:Node 1: stage 0 completes. uploading public keys...
DEBUG:flwr:Node 2: starting stage 0...
DEBUG:flwr:Node 2: stage 0 completes. uploading public keys...
DEBUG:flwr:Node 3: starting stage 0...
DEBUG:flwr:Node 3: st

Saved: schedule_sweep_v19.json
Saved: schedule_sweep_v19.json


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved: dp_budget_composition_v19.json
Sweep complete: 18 evaluations; 27 paired comparisons


In [13]:
# Table 3-D: centralised/federated gap and DP cost by schedule length — generated automatically from schedule_sweep_v19.json
def fmt(s,scale=1.,digits=2,sd=True):
    if not s or s.get('mean') is None:return '—'
    txt=f'{s["mean"]*scale:.{digits}f}'
    if sd and s.get('sd') is not None:txt+=f' ± {s["sd"]*scale:.{digits}f}'
    return txt
S=sweep_result['summary_by_rounds'];T=sweep_result['equivalence_central_vs_fedavg_by_rounds']
CASE_AR={'Central':'Central','A_FedAvg':'A: FedAvg','E_Clipped_Poisson_no_noise':'E: clipping + Poisson','D_DP_SecAgg_e3':'D: DP + SecAgg (ε = 3)'}
lines=[f'# Table 3-D — schedule-length sweep ({Path(RUN_DIR).name})','',f'Pre-specified question: {SWEEP["question"]}','',
       '## Accuracy (%) and AUC under the common threshold policy, by number of rounds (mean ± SD over seeds)','',
       '| Rounds | '+' | '.join(f'{CASE_AR[c]} accuracy' for c in CASE_AR)+' | '+' | '.join(f'{CASE_AR[c]} AUC' for c in CASE_AR)+' |',
       '|---|'+'---:|'*(2*len(CASE_AR))]
for R in [str(r) for r in SWEEP['rounds']]+['best']:
    row=S.get(R,{})
    lab='best validation round' if R=='best' else R
    lines.append(f'| {lab} | '+' | '.join(fmt(row.get(c,{}).get('validation_recall',{}).get('acc'),100) for c in CASE_AR)+' | '+' | '.join(fmt(row.get(c,{}).get('validation_recall',{}).get('auc'),1,4) for c in CASE_AR)+' |')
lines+=['','## Central − federated gap and corrected equivalence test (common threshold policy)','','| Rounds | Accuracy difference | 90% CI | Equivalent (margin 0.01)? | AUC difference | 90% CI | Equivalent (margin 0.005)? |','|---|---:|---|---|---:|---|---|']
for R in [str(r) for r in SWEEP['rounds']]+['best']:
    t=T.get(R,{}).get('validation_recall',{});a=t.get('acc',{});u=t.get('auc',{})
    def cell(x):return (f'{x["mean_diff"]:.4f}',f'[{x["ci90"][0]:.4f}, {x["ci90"][1]:.4f}]',str(x.get('equivalent'))) if x.get('ci90') else ('—','—',x.get('status','—'))
    ca=cell(a);cu=cell(u);lab='best validation round' if R=='best' else R
    lines.append(f'| {lab} | {ca[0]} | {ca[1]} | {ca[2]} | {cu[0]} | {cu[1]} | {cu[2]} |')
lines+=['','## Privacy cost by schedule length (A − D, paired accuracy difference, mean over seeds)','','| Rounds | A − D (accuracy) | A − E (accuracy: clipping) | E − D (accuracy: noise) | recorded ε of the maximum client |','|---|---:|---:|---:|---:|']
for R in SWEEP['rounds']:
    def md(ca,cb):
        v=[p['metrics']['accuracy']['difference'] for p in sweep_result['paired_comparisons'] if p['rounds']==R and p['first_case']==ca and p['second_case']==cb and p['policy']=='validation_recall' and p['status']=='complete']
        return f'{100*np.mean(v):.2f}' if v else '—'
    eps=[r.get('max_client_epsilon_reported') for r in sweep_result['records'] if r['case']=='D_DP_SecAgg_e3' and r['rounds']==R]
    lines.append(f'| {R} | {md("A_FedAvg","D_DP_SecAgg_e3")} | {md("A_FedAvg","E_Clipped_Poisson_no_noise")} | {md("E_Clipped_Poisson_no_noise","D_DP_SecAgg_e3")} | {np.mean([e for e in eps if e is not None]):.3f} |' if any(e is not None for e in eps) else f'| {R} | {md("A_FedAvg","D_DP_SecAgg_e3")} | {md("A_FedAvg","E_Clipped_Poisson_no_noise")} | {md("E_Clipped_Poisson_no_noise","D_DP_SecAgg_e3")} | — |')
lines+=['',(f'Composed ε over all DP releases in this sweep (maximum college) = {composition["epsilon_max_composed"]:.2f} at δ = {composition["delta"]}.' if composition.get('epsilon_max_composed') is not None else 'No DP releases in this sweep; no privacy budget consumed.')]
tables_path=output_path('paper_tables','.md');tables_path.write_text('\n'.join(lines),encoding='utf-8');print('\n'.join(lines))


# Table 3-D — schedule-length sweep (run_20260910_190700_v19_c668991d)

Pre-specified question: At 50 rounds, what is the directly measured DP cost (A minus D) and does it match the clipping-only control (A minus E, E minus D) as it did at 5-25 rounds?

## Accuracy (%) and AUC under the common threshold policy, by number of rounds (mean ± SD over seeds)

| Rounds | Central accuracy | A: FedAvg accuracy | E: clipping + Poisson accuracy | D: DP + SecAgg (ε = 3) accuracy | Central AUC | A: FedAvg AUC | E: clipping + Poisson AUC | D: DP + SecAgg (ε = 3) AUC |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 50 | 89.66 ± 0.20 | 89.49 ± 0.08 | 87.77 ± 0.19 | 87.75 ± 0.22 | 0.9481 ± 0.0012 | 0.9447 ± 0.0015 | 0.9323 ± 0.0015 | 0.9326 ± 0.0016 |
| best validation round | 89.67 ± 0.22 | 89.44 ± 0.19 | 87.67 ± 0.23 | — | 0.9482 ± 0.0011 | 0.9447 ± 0.0015 | 0.9324 ± 0.0015 | — |

## Central − federated gap and corrected equivalence test (common threshold policy)

| Rounds | Accuracy difference | 9

In [14]:
import zipfile
try:
    history=get_ipython().history_manager.input_hist_raw
except (NameError,AttributeError):history=[]
save_result('execution_source_audit',{'generated_core_sha256':NOTEBOOK_CORE_SHA256,
 'executed_input_sha256':hashlib.sha256('\n'.join(history).encode()).hexdigest() if history else None,
 'note':'Generated code fingerprint and executed input-history fingerprint are distinct; no source history or secrets exported.'})
files=sorted(p for p in Path(RUN_DIR).iterdir() if p.is_file() and '_v19.' in p.name and p.name!='artifact_inventory_v19.json')
records=[{'name':p.name,'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in files]
save_result('artifact_inventory',{'run_mode':RUN_MODE,'files':records,
 'scope':'All versioned result files except this inventory. Private data/model shards are excluded.'})
for r in records:
    p=Path(RUN_DIR)/r['name']
    assert p.stat().st_size==r['bytes'] and hashlib.sha256(p.read_bytes()).hexdigest()==r['sha256']
archive=Path(RUN_DIR).with_suffix('.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(Path(RUN_DIR).iterdir()):
        if p.is_file() and '_v19.' in p.name:z.write(p,arcname=Path(RUN_DIR).name+'/'+p.name)
print('Result directory:',RUN_DIR)
print('Results archive:',archive)
try:
    from IPython.display import display,FileLink
    display(FileLink(str(archive)))
except ImportError:pass


Saved: execution_source_audit_v19.json
Saved: artifact_inventory_v19.json
Result directory: /content/drive/MyDrive/iudb/results/run_20260910_190700_v19_c668991d
Results archive: /content/drive/MyDrive/iudb/results/run_20260910_190700_v19_c668991d.zip


/content/drive/MyDrive/iudb/results/run_20260910_190700_v19_c668991d.zip

## README — v19

Purpose: measure the cost of DP + SecAgg+ (ε = 3) directly at 50 rounds, completing the last row of Table 3-D, which previously relied on the control arm E alone.

Design: arms Central, A and E are repeated at 50 rounds (deterministic repeat of v18; any numerical difference is treated as an error) and D is trained at 50 rounds with three seeds. The comparisons and tests are those used in v17/v18.

Quantities reported: A − D at 50 rounds (previously inferred ≈ 1.7 points from A − E), E − D (expected ≤ 0.1 points as at 5–25 rounds), the recorded ε of the maximum client, and the composed ε of this run.

Note: this run consumes an additional privacy budget (three DP releases per college) that adds to the composed budget reported in Section 5.6.
